# Phase 8 — AI Retail Assistant & Product Knowledge Layer

## Objective

The objective of this phase is to build a grounded AI-powered retail
assistant that converts the governed outputs of the Smart AI Retail
System into natural-language business insights and product knowledge.

The assistant will support two primary use cases:

### 1. Retail Analytics Assistant
Enable users to ask business questions relating to:

- Sales performance
- Revenue and profitability
- Inventory position
- Product velocity
- ABC classification
- Reorder recommendations
- Slow-moving products
- Forecasting outputs
- SOA information
- Competitor pricing
- Pricing recommendations

All analytical answers must be generated from governed project data
rather than raw source files or unsupported model knowledge.

### 2. Product Knowledge Assistant
Enable retail staff to search a product using its model number
(and later barcode/QR code) and retrieve:

- Product identity
- Category
- Description
- Key specifications
- Dimensions
- Features
- Manufacturer information
- Source documentation
- Comparative product advantages/disadvantages

Product specifications must only be returned when the product can be
confidently matched to a verified source.

## Core Design Principle

> Retrieve → Validate → Ground → Answer

The AI assistant must never invent business metrics or product
specifications.

If reliable information is unavailable, the system should explicitly
return that the information could not be confidently determined.

## Expected Phase 8 Output

By the end of this phase the system should provide:

1. A governed retail-data query layer
2. An approved business-question set
3. Product-model lookup and matching
4. Structured product knowledge records
5. Grounded product comparisons
6. Natural-language retail Q&A
7. Source and refresh-date attribution
8. A staff-facing product knowledge interface
9. Grounding and hallucination test results

### Step 1 — Governed AI Data Layer

The AI Retail Assistant will not access raw operational Excel files.

Instead, it will consume governed analytical datasets created during
the earlier phases of the project.

### Core Data Sources

- `dim_product` — canonical product information
- `fact_sales` — governed sales and profitability data
- `fact_stock` — store-level inventory
- `fact_soa` — Sell Out Allowance periods and values
- `inventory_base` — inventory intelligence and recommendations

### Extended Intelligence Sources

- Forecasting output from Phase 6
- Competitor pricing and pricing recommendations from Phase 7

These datasets collectively form the read-only knowledge layer used by
the AI Retail Assistant.

The assistant must retrieve factual values from these governed datasets
before generating a natural-language response.

Raw Excel source files will not be queried directly.

In [1]:
##  Load Governed Core Datasets

import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Project paths
# --------------------------------------------------
PROCESSED_DIR = Path("../data/processed")

# --------------------------------------------------
# Load governed datasets
# --------------------------------------------------

PROJECT_ROOT = Path.cwd().parent.parent

DIM_PRODUCT_PATH = PROJECT_ROOT / "data" / "processed" / "dim_product.csv"
SALES_PATH = PROJECT_ROOT / "data" / "processed" / "fact_sales.csv"
SOA_PATH  = PROJECT_ROOT / "data" / "processed" / "fact_soa.csv"
STOCK_PATH = PROJECT_ROOT / "data" / "processed" / "fact_stock.csv"
INVENTORY_PATH = PROJECT_ROOT / "data" / "processed" / "phase5_inventory_intelligence"/"inventory_decision_layer.csv"

dim_product = pd.read_csv(DIM_PRODUCT_PATH)
fact_sales = pd.read_csv(SALES_PATH)
fact_soa = pd.read_csv(SOA_PATH)
fact_stock = pd.read_csv(STOCK_PATH)

inventory_decision = pd.read_csv(INVENTORY_PATH)

print("Governed Phase 8 datasets loaded successfully.")

Governed Phase 8 datasets loaded successfully.


In [2]:
datasets = {
    "dim_product": dim_product,
    "fact_sales": fact_sales,
    "fact_stock": fact_stock,
    "fact_soa": fact_soa,
    "inventory_decision": inventory_decision
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 60)
    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())


dim_product
------------------------------------------------------------
Shape: (1436, 18)
Columns:
['Product_ID', 'Product_Key', 'Product_Description', 'Product_Category', 'Record_Type', 'Source_Status', 'Source_Count', 'In_Sales', 'In_Stock', 'In_SOA', 'Sales_Stock_Code', 'Sales_Description', 'Sales_Category', 'Stock_Model', 'Stock_Description', 'Stock_Category', 'SOA_Model', 'SOA_Description']

fact_sales
------------------------------------------------------------
Shape: (966, 18)
Columns:
['Sales_Record_ID', 'Product_ID', 'Product_Key', 'Source_Month', 'Category', 'Stock Code', 'Description', 'Record_Type', 'Level', 'Sold Period', 'Transaction_Status', 'Unit Cost', 'Unit Price', 'Cost Sales', 'Sales Value', 'Profit', 'Profit %', 'Cost_Sales_Reconciliation_Flag']

fact_stock
------------------------------------------------------------
Shape: (2681, 9)
Columns:
['Product_ID', 'Model', 'Store', 'Category', 'Description', 'Quantity', 'Stock_Status', 'Outstanding_Order_Qty', 'store_id

In [3]:
for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

dim_product: 1,436 rows × 18 columns
fact_sales: 966 rows × 18 columns
fact_stock: 2,681 rows × 9 columns
fact_soa: 412 rows × 11 columns
inventory_decision: 2,681 rows × 27 columns


In [4]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("=" * 80)
    print(df.columns.tolist())


dim_product
['Product_ID', 'Product_Key', 'Product_Description', 'Product_Category', 'Record_Type', 'Source_Status', 'Source_Count', 'In_Sales', 'In_Stock', 'In_SOA', 'Sales_Stock_Code', 'Sales_Description', 'Sales_Category', 'Stock_Model', 'Stock_Description', 'Stock_Category', 'SOA_Model', 'SOA_Description']

fact_sales
['Sales_Record_ID', 'Product_ID', 'Product_Key', 'Source_Month', 'Category', 'Stock Code', 'Description', 'Record_Type', 'Level', 'Sold Period', 'Transaction_Status', 'Unit Cost', 'Unit Price', 'Cost Sales', 'Sales Value', 'Profit', 'Profit %', 'Cost_Sales_Reconciliation_Flag']

fact_stock
['Product_ID', 'Model', 'Store', 'Category', 'Description', 'Quantity', 'Stock_Status', 'Outstanding_Order_Qty', 'store_id']

fact_soa
['SOA_Record_ID', 'Product_ID', 'Product_Key', 'Model', 'Description', 'Starts', 'Ends', 'Window_Days', 'SOA', 'Original_Ends', 'Date_Correction_Flag']

inventory_decision
['Product_ID', 'Model', 'Store', 'Category', 'Description', 'Quantity', 'Stoc

In [5]:
print("fact_sales shape:", fact_sales.shape)
print("fact_stock shape:", fact_stock.shape)
print("fact_soa shape:", fact_soa.shape)

print("\nFACT SALES SAMPLE")
display(fact_sales.head())

print("\nFACT STOCK SAMPLE")
display(fact_stock.head())

print("\nFACT SOA SAMPLE")
display(fact_soa.head())

fact_sales shape: (966, 18)
fact_stock shape: (2681, 9)
fact_soa shape: (412, 11)

FACT SALES SAMPLE


,Sales_Record_ID,Product_ID,Product_Key,Source_Month,Category,Stock Code,Description,Record_Type,Level,Sold Period,Transaction_Status,Unit Cost,Unit Price,Cost Sales,Sales Value,Profit,Profit %,Cost_Sales_Reconciliation_Flag
0,1,1223,TLS169BOXE,Nov,ACCESSORIES,TLS169BOXE,Boxed Uni Floor Tool 30-38MM,PRODUCT,0,1,POSITIVE_SALES_ACTIVITY,9.31,29.99,9.31,24.99,15.68,62.75,MATCH
1,2,17,112.204,Nov,ACCESSORIES,112.204,TV Arial Lead 4.0m,PRODUCT,3,1,POSITIVE_SALES_ACTIVITY,1.28,2.99,1.28,2.49,1.21,48.59,MATCH
2,3,385,DLSC500,Nov,ACCESSORIES,DLSC500,Delonghi Descaler,PRODUCT,5,1,POSITIVE_SALES_ACTIVITY,7.77,14.49,7.77,7.50,-0.27,-3.60,MATCH
3,4,246,AF01,Nov,ACCESSORIES,AF01,VACUUM FRESHENERS AF101,PRODUCT,3,2,POSITIVE_SALES_ACTIVITY,2.30,3.49,4.60,3.32,-1.28,-38.55,MATCH
4,5,1076,SES007NEU0,Nov,ACCESSORIES,SES007NEU0,Sage Descaler (pack of 4),PRODUCT,6,2,POSITIVE_SALES_ACTIVITY,9.39,14.49,18.78,23.32,4.54,19.47,MATCH



FACT STOCK SAMPLE


,Product_ID,Model,Store,Category,Description,Quantity,Stock_Status,Outstanding_Order_Qty,store_id
0,221,980531,Belfast,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,2,IN_STOCK,0,1
1,221,980531,Blanch,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0,2
2,221,980531,Cavan,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0,3
3,221,980531,Dundrum,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,1,IN_STOCK,0,4
4,221,980531,Gorey,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,0,OUT_OF_STOCK,0,5



FACT SOA SAMPLE


,SOA_Record_ID,Product_ID,Product_Key,Model,Description,Starts,Ends,Window_Days,SOA,Original_Ends,Date_Correction_Flag
0,1,11,107833-01,107833-01,Dyson Supersonic Hair Dryer,31/12/2025,03/02/2026,35,66.50,03/02/2026,False
1,2,36,161818-01,161818-01,Dyson Supersonic Ceramic,31/12/2025,03/02/2026,35,66.50,03/02/2026,False
2,3,39,19750,19750,Russell Hobbs Rice Cooker 1.8Ltr,01/02/2026,28/02/2026,28,5.00,28/02/2026,False
3,4,48,21270,21270,Russell Hobbs White Textures Jug,01/02/2026,28/02/2026,28,2.29,28/02/2026,False
4,5,49,21271,21271,Russell Hobbs Black Textures Jug,01/02/2026,28/02/2026,28,2.29,28/02/2026,False


In [6]:
print("fact_sales == fact_stock columns:",
      fact_sales.columns.tolist() == fact_stock.columns.tolist())

print(
    "fact_sales == fact_stock shape:",
    fact_sales.shape == fact_stock.shape
)

fact_sales == fact_stock columns: False
fact_sales == fact_stock shape: False


In [7]:
print("Unique Product_IDs")
print("fact_sales:", fact_sales["Product_ID"].nunique())
print("fact_stock:", fact_stock["Product_ID"].nunique())
print("fact_soa:", fact_soa["Product_ID"].nunique())

Unique Product_IDs
fact_sales: 767
fact_stock: 383
fact_soa: 412


In [8]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("=" * 80)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())


dim_product
Shape: (1436, 18)
Columns: ['Product_ID', 'Product_Key', 'Product_Description', 'Product_Category', 'Record_Type', 'Source_Status', 'Source_Count', 'In_Sales', 'In_Stock', 'In_SOA', 'Sales_Stock_Code', 'Sales_Description', 'Sales_Category', 'Stock_Model', 'Stock_Description', 'Stock_Category', 'SOA_Model', 'SOA_Description']

fact_sales
Shape: (966, 18)
Columns: ['Sales_Record_ID', 'Product_ID', 'Product_Key', 'Source_Month', 'Category', 'Stock Code', 'Description', 'Record_Type', 'Level', 'Sold Period', 'Transaction_Status', 'Unit Cost', 'Unit Price', 'Cost Sales', 'Sales Value', 'Profit', 'Profit %', 'Cost_Sales_Reconciliation_Flag']

fact_stock
Shape: (2681, 9)
Columns: ['Product_ID', 'Model', 'Store', 'Category', 'Description', 'Quantity', 'Stock_Status', 'Outstanding_Order_Qty', 'store_id']

fact_soa
Shape: (412, 11)
Columns: ['SOA_Record_ID', 'Product_ID', 'Product_Key', 'Model', 'Description', 'Starts', 'Ends', 'Window_Days', 'SOA', 'Original_Ends', 'Date_Correction

## Step 2 — AI Data Dictionary & Approved Question Set

The AI Retail Assistant must answer questions only from governed datasets.

Each dataset has a defined analytical responsibility:

### Product Master — `dim_product`
Authoritative source for:
- Product_ID
- Product_Key
- Product description
- Product category
- Source presence
- Sales / Stock / SOA model mappings

### Sales — `fact_sales`
Authoritative source for:
- Units sold
- Revenue
- Unit cost
- Unit price
- Cost of sales
- Profit
- Profit percentage
- Sales period
- Sales category
- Transaction status

### Stock — `fact_stock`
Authoritative source for:
- Current stock quantity
- Store-level stock
- Stock status
- Outstanding order quantity

### SOA — `fact_soa`
Authoritative source for:
- SOA value
- SOA start date
- SOA end date
- SOA window duration
- Date correction status

### Inventory Intelligence — `inventory_decision`
Authoritative source for:
- Sales velocity
- ABC classification
- Demand status
- Stock band
- Reorder recommendation
- Slow-mover action
- Inventory decision
- Decision priority
- Inventory segment

The assistant must not infer unavailable metrics from unrelated tables.
If a requested metric cannot be supported by the governed data,
the assistant must explicitly report that the information is unavailable.

In [9]:
approved_question_map = {
    "sales_performance": {
        "dataset": "fact_sales",
        "examples": [
            "What are the top-selling products?",
            "Which products generated the highest revenue?",
            "Which category has the highest profit?",
            "What is the margin for a product?",
            "How many units were sold for a model?"
        ]
    },

    "stock_position": {
        "dataset": "fact_stock",
        "examples": [
            "How much stock do we have for a product?",
            "Which store has the most stock?",
            "Which products are out of stock?",
            "What outstanding orders exist for a product?"
        ]
    },

    "soa_information": {
        "dataset": "fact_soa",
        "examples": [
            "What is the SOA for a model?",
            "When does the current SOA start?",
            "When does the SOA expire?",
            "How long is the SOA window?"
        ]
    },

    "inventory_intelligence": {
        "dataset": "inventory_decision",
        "examples": [
            "Which products need reordering?",
            "Which products are slow movers?",
            "Which products are high demand but low stock?",
            "Which products require urgent inventory action?",
            "What is the ABC class of a product?"
        ]
    },

    "product_lookup": {
        "dataset": "dim_product",
        "examples": [
            "What category is this product?",
            "What is the product description?",
            "What is the canonical Product_ID?",
            "Is this product present in sales, stock and SOA?"
        ]
    }
}

In [10]:
for topic, config in approved_question_map.items():
    print(f"\n{topic.upper()}")
    print("Dataset:", config["dataset"])

    for question in config["examples"]:
        print("-", question)


SALES_PERFORMANCE
Dataset: fact_sales
- What are the top-selling products?
- Which products generated the highest revenue?
- Which category has the highest profit?
- What is the margin for a product?
- How many units were sold for a model?

STOCK_POSITION
Dataset: fact_stock
- How much stock do we have for a product?
- Which store has the most stock?
- Which products are out of stock?
- What outstanding orders exist for a product?

SOA_INFORMATION
Dataset: fact_soa
- What is the SOA for a model?
- When does the current SOA start?
- When does the SOA expire?
- How long is the SOA window?

INVENTORY_INTELLIGENCE
Dataset: inventory_decision
- Which products need reordering?
- Which products are slow movers?
- Which products are high demand but low stock?
- Which products require urgent inventory action?
- What is the ABC class of a product?

PRODUCT_LOOKUP
Dataset: dim_product
- What category is this product?
- What is the product description?
- What is the canonical Product_ID?
- Is thi

In [11]:
## Add authority rules
field_authority = {
    "product_identity": "dim_product",
    "product_category": "dim_product",
    "product_description": "dim_product",

    "units_sold": "fact_sales",
    "revenue": "fact_sales",
    "profit": "fact_sales",
    "margin": "fact_sales",
    "unit_cost": "fact_sales",
    "unit_price": "fact_sales",

    "stock_quantity": "fact_stock",
    "stock_status": "fact_stock",
    "store": "fact_stock",
    "outstanding_order_qty": "fact_stock",

    "soa_value": "fact_soa",
    "soa_start": "fact_soa",
    "soa_end": "fact_soa",
    "soa_window": "fact_soa",

    "velocity": "inventory_decision",
    "abc_class": "inventory_decision",
    "reorder_recommendation": "inventory_decision",
    "slow_mover_action": "inventory_decision",
    "inventory_decision": "inventory_decision",
    "decision_priority": "inventory_decision"
}

### Step 4 — Product Lookup & Model Matching Engine

The objective of this step is to reliably resolve a staff-entered model
number to the canonical product record stored in `dim_product`.

The matching process follows three principles:

1. Normalize model-number formatting before comparison.
2. Prefer exact model matches across governed source fields.
3. Never return a guessed product when confidence is low.

Only high-confidence matches should proceed automatically to the
Product Knowledge Layer.

In [13]:
import re 
def normalize_model(value):
    """
    Normalize model numbers for comparison.

    Rules:
    - convert to uppercase
    - remove spaces
    - remove hyphens
    - remove underscores
    - remove other non-alphanumeric characters
    """

    if pd.isna(value):
        return None

    value = str(value).upper().strip()
    value = re.sub(r"[^A-Z0-9]", "", value)

    return value


model_columns = [
    "Product_Key",
    "Sales_Stock_Code",
    "Stock_Model",
    "SOA_Model"
]

for col in model_columns:
    dim_product[f"{col}_Normalized"] = (
        dim_product[col]
        .apply(normalize_model)
    )

print("Normalized model fields created.")

Normalized model fields created.


In [14]:
def lookup_product(model_number, product_df=dim_product):

    query = normalize_model(model_number)

    if not query:
        return {
            "status": "invalid_input",
            "match_confidence": "none",
            "message": "No valid model number was provided."
        }

    match_columns = [
        "Product_Key_Normalized",
        "Sales_Stock_Code_Normalized",
        "Stock_Model_Normalized",
        "SOA_Model_Normalized"
    ]

    mask = False

    for col in match_columns:
        mask = mask | (product_df[col] == query)

    matches = product_df[mask].copy()

    if len(matches) == 0:
        return {
            "status": "not_matched",
            "match_confidence": "none",
            "message": "Product could not be confidently matched."
        }

    if len(matches) > 1:
        return {
            "status": "multiple_matches",
            "match_confidence": "medium",
            "match_count": len(matches),
            "matches": matches[
                [
                    "Product_ID",
                    "Product_Key",
                    "Product_Description",
                    "Product_Category"
                ]
            ]
        }

    row = matches.iloc[0]

    return {
        "status": "matched",
        "match_confidence": "high",
        "Product_ID": row["Product_ID"],
        "Product_Key": row["Product_Key"],
        "Product_Description": row["Product_Description"],
        "Product_Category": row["Product_Category"],
        "In_Sales": row["In_Sales"],
        "In_Stock": row["In_Stock"],
        "In_SOA": row["In_SOA"]
    }


test_model = dim_product["Product_Key"].dropna().iloc[0]

print("Testing model:", test_model)

result = lookup_product(test_model)

result

Testing model: 010-02384-10


{'status': 'matched',
 'match_confidence': 'high',
 'Product_ID': np.int64(1),
 'Product_Key': '010-02384-10',
 'Product_Description': 'Garmin Lily Cream Gold & White',
 'Product_Category': 'FITNESS',
 'In_Sales': np.True_,
 'In_Stock': np.False_,
 'In_SOA': np.False_}

### Step 5 — Unified Product Retail Profile

After a product is confidently matched, the assistant should retrieve
all available governed information for that Product_ID.

The unified profile combines:

- Product identity
- Sales performance
- Current stock position
- SOA information
- Inventory intelligence

This creates a single governed context object that can later be used by
the AI assistant to answer natural-language questions.

In [15]:
def build_product_profile(product_id):

    profile = {
        "Product_ID": int(product_id)
    }

    # --------------------------------------------------
    # Product master
    # --------------------------------------------------

    product_row = dim_product[
        dim_product["Product_ID"] == product_id
    ]

    if not product_row.empty:
        row = product_row.iloc[0]

        profile["Product"] = {
            "Product_Key": row["Product_Key"],
            "Description": row["Product_Description"],
            "Category": row["Product_Category"],
            "In_Sales": bool(row["In_Sales"]),
            "In_Stock": bool(row["In_Stock"]),
            "In_SOA": bool(row["In_SOA"])
        }

    # --------------------------------------------------
    # Sales
    # --------------------------------------------------

    sales_rows = fact_sales[
        fact_sales["Product_ID"] == product_id
    ]

    if not sales_rows.empty:

        profile["Sales"] = {
            "Records": int(len(sales_rows)),
            "Units_Sold": float(
                sales_rows["Sold Period"].fillna(0).sum()
            ),
            "Revenue": float(
                sales_rows["Sales Value"].fillna(0).sum()
            ),
            "Profit": float(
                sales_rows["Profit"].fillna(0).sum()
            )
        }

    else:
        profile["Sales"] = None

    # --------------------------------------------------
    # Stock
    # --------------------------------------------------

    stock_rows = fact_stock[
        fact_stock["Product_ID"] == product_id
    ]

    if not stock_rows.empty:

        profile["Stock"] = {
            "Total_Quantity": float(
                stock_rows["Quantity"].fillna(0).sum()
            ),

            "Stores": stock_rows[
                [
                    "Store",
                    "Quantity",
                    "Stock_Status",
                    "Outstanding_Order_Qty"
                ]
            ].to_dict("records")
        }

    else:
        profile["Stock"] = None

    # --------------------------------------------------
    # SOA
    # --------------------------------------------------

    soa_rows = fact_soa[
        fact_soa["Product_ID"] == product_id
    ]

    if not soa_rows.empty:

        profile["SOA"] = soa_rows[
            [
                "Model",
                "Starts",
                "Ends",
                "Window_Days",
                "SOA"
            ]
        ].to_dict("records")

    else:
        profile["SOA"] = None

    # --------------------------------------------------
    # Inventory intelligence
    # --------------------------------------------------

    inv_rows = inventory_decision[
        inventory_decision["Product_ID"] == product_id
    ]

    if not inv_rows.empty:

        profile["Inventory_Intelligence"] = inv_rows[
            [
                "Store",
                "ABC_Class",
                "Velocity_Band",
                "Demand_Status",
                "Stock_Band",
                "Reorder_Recommendation",
                "Slow_Mover_Action",
                "Inventory_Decision",
                "Decision_Priority",
                "Inventory_Segment"
            ]
        ].to_dict("records")

    else:
        profile["Inventory_Intelligence"] = None

    return profile


matched = lookup_product("010-02384-10")

product_profile = build_product_profile(
    matched["Product_ID"]
)

product_profile

{'Product_ID': 1,
 'Product': {'Product_Key': '010-02384-10',
  'Description': 'Garmin Lily Cream Gold & White',
  'Category': 'FITNESS',
  'In_Sales': True,
  'In_Stock': False,
  'In_SOA': False},
 'Sales': {'Records': 1,
  'Units_Sold': 1.0,
  'Revenue': 165.83,
  'Profit': 41.86},
 'Stock': None,
 'SOA': None,
 'Inventory_Intelligence': None}

### Step 6 — Natural-Language Answer Layer

This step converts governed product-profile data into concise,
human-readable retail answers.

The answer layer must:

- Use only values present in the governed profile
- Clearly distinguish available and unavailable information
- Avoid estimating or inventing missing metrics
- Preserve Product_ID/model grounding
- Produce consistent responses suitable for later use in a chat UI

In [17]:
def generate_product_summary(profile):

    product = profile.get("Product", {})
    sales = profile.get("Sales")
    stock = profile.get("Stock")
    soa = profile.get("SOA")
    inventory = profile.get("Inventory_Intelligence")

    lines = []

    # Product identity
    lines.append(
        f"{product.get('Description')} "
        f"({product.get('Product_Key')}) "
        f"is in the {product.get('Category')} category."
    )

    # Sales
    if sales:
        lines.append(
            f"It has sold {sales['Units_Sold']:.0f} unit(s), "
            f"generating £{sales['Revenue']:.2f} in revenue "
            f"and £{sales['Profit']:.2f} in profit."
        )
    else:
        lines.append(
            "No governed sales information is available for this product."
        )

    # Stock
    if stock:
        lines.append(
            f"Current total governed stock is "
            f"{stock['Total_Quantity']:.0f} unit(s)."
        )
    else:
        lines.append(
            "No governed stock information is available for this product."
        )

    # SOA
    if soa:
        lines.append(
            f"{len(soa)} SOA record(s) are available for this product."
        )
    else:
        lines.append(
            "No governed SOA information is available for this product."
        )

    # Inventory intelligence
    if inventory:
        lines.append(
            "Inventory intelligence recommendations are available."
        )
    else:
        lines.append(
            "No inventory intelligence recommendation is available."
        )

    return " ".join(lines)


summary = generate_product_summary(product_profile)

print(summary)

Garmin Lily Cream Gold & White (010-02384-10) is in the FITNESS category. It has sold 1 unit(s), generating £165.83 in revenue and £41.86 in profit. No governed stock information is available for this product. No governed SOA information is available for this product. No inventory intelligence recommendation is available.


In [18]:
def ask_product(model_number):

    match = lookup_product(model_number)

    if match["status"] != "matched":
        return match["message"]

    profile = build_product_profile(
        match["Product_ID"]
    )

    return generate_product_summary(profile)


print(
    ask_product("010-02384-10")
)

Garmin Lily Cream Gold & White (010-02384-10) is in the FITNESS category. It has sold 1 unit(s), generating £165.83 in revenue and £41.86 in profit. No governed stock information is available for this product. No governed SOA information is available for this product. No inventory intelligence recommendation is available.


### Step 7 — Business Question Functions

This step introduces governed analytical functions for common
management and operational questions.

The first supported question groups are:

- Top-selling products
- Highest-revenue products
- Highest-profit products
- Category performance
- Reorder candidates
- Slow-moving products
- High-priority inventory actions
- Active SOA records

Each question is answered from its authoritative governed dataset.

In [19]:
def top_selling_products(n=10):

    result = (
        fact_sales.groupby(
            ["Product_ID", "Stock Code", "Description", "Category"],
            dropna=False
        )["Sold Period"]
        .sum()
        .reset_index()
        .sort_values("Sold Period", ascending=False)
        .head(n)
    )

    return result

def top_revenue_products(n=10):

    result = (
        fact_sales.groupby(
            ["Product_ID", "Stock Code", "Description", "Category"],
            dropna=False
        )["Sales Value"]
        .sum()
        .reset_index()
        .sort_values("Sales Value", ascending=False)
        .head(n)
    )

    return result

def top_profit_products(n=10):

    result = (
        fact_sales.groupby(
            ["Product_ID", "Stock Code", "Description", "Category"],
            dropna=False
        )["Profit"]
        .sum()
        .reset_index()
        .sort_values("Profit", ascending=False)
        .head(n)
    )

    return result

def category_performance():

    result = (
        fact_sales.groupby("Category", dropna=False)
        .agg(
            Units_Sold=("Sold Period", "sum"),
            Revenue=("Sales Value", "sum"),
            Profit=("Profit", "sum")
        )
        .reset_index()
        .sort_values("Revenue", ascending=False)
    )

    return result

def reorder_candidates():

    return inventory_decision[
        inventory_decision["Reorder_Recommendation"]
        .astype(str)
        .str.contains("REORDER", case=False, na=False)
    ].copy()

def slow_movers():

    return inventory_decision[
        inventory_decision["Slow_Mover_Action"]
        .astype(str)
        .str.contains(
            "review|markdown|promotion|slow",
            case=False,
            na=False
        )
    ].copy()

def high_priority_inventory():

    return inventory_decision[
        inventory_decision["Decision_Priority"]
        .astype(str)
        .str.contains("HIGH", case=False, na=False)
    ].copy()



In [20]:
fact_soa["Starts"] = pd.to_datetime(
    fact_soa["Starts"],
    errors="coerce"
)

fact_soa["Ends"] = pd.to_datetime(
    fact_soa["Ends"],
    errors="coerce"
)

def active_soa(as_of_date=None):

    if as_of_date is None:
        as_of_date = pd.Timestamp.today().normalize()
    else:
        as_of_date = pd.to_datetime(as_of_date)

    return fact_soa[
        (fact_soa["Starts"] <= as_of_date)
        &
        (fact_soa["Ends"] >= as_of_date)
    ].copy()

C:\Users\singh\AppData\Local\Temp\ipykernel_41964\2464617959.py:1: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  fact_soa["Starts"] = pd.to_datetime(


In [21]:
display(top_selling_products(10))

,Product_ID,Stock Code,Description,Category,Sold Period
383,615,INSTALLATIO,INSTALLATION FEE,INSTALLATION,27
638,1167,T2351V11,Eufy Robot Vaccum X10 Pro Omni,ROBOT CLEANING,16
454,767,M,MISCELANEOUS,MISC,12
237,369,Delivery,WEB DELIVERY CHARGE,DELIVERY CHARGE,9
706,1289,VS15A6031R4Samsung,Jet 60 Cordless Vacuum,STICK VACS,9
103,121,43LQ60006LA.LG,"43"" Smart TV",TV 33 - 43,8
517,886,P-SDU32GU18PNY,Elite microSDHC card 32G,IT ACCESSORIES,8
328,525,GN BAGS,BAGS 400/600/800 SERIES AND S5,VACUUM BAGS,7
238,370,DELIVERY-CHLOCAL,DELIVERY CHARGE,DELIVERY CHARGE,7
437,721,KN650A,Kenwood Electric Knife | KN650A,FOOD PREP,7


In [22]:
display(top_revenue_products(10))

,Product_ID,Stock Code,Description,Category,Sales Value
638,1167,T2351V11,Eufy Robot Vaccum X10 Pro Omni,ROBOT CLEANING,6818.30
98,116,42120,Novy Panorama 120 Pro 5 Zone,DOWNDRAFT HOBS,4000.00
512,870,OLED65G54L,LG 65in G5 OLED TV,TV 60 - 70,3165.00
715,1320,WEK365WCS,Miele (12392780) 10kg 1400 Spin,WASHING MACHINES,2832.49
660,1213,TEH785WP,Miele 11871830 9kg Heat Pump,TUMBLE DRYERS,2764.17
350,560,H7464BPBL,Miele Black 11093600 Pyro Single,SINGLE OVENS,2683.34
764,1425,XRFSD5265,Liebherr SXS,FRIDGE FREEZERS,2666.67
507,856,OLED55G54L,LG 55in G5 OLED TV,TV 51 - 59,2581.67
570,1037,RS70F64KEF,Samsung Black St/St USA FF,USA F/F,2580.83
682,1246,U1ACE2AG3BNeff,N50 Graphite Double Oven,DOUBLE OVENS,2538.33


In [23]:
display(category_performance())

,Category,Units_Sold,Revenue,Profit
85,WASHING MACHINES,50,22279.54,1261.05
72,TV 51 - 59,33,17919.17,-3493.90
80,USA F/F,19,17525.42,-371.89
56,SINGLE OVENS,29,17204.18,1678.75
67,TUMBLE DRYERS,28,15132.28,2169.38
...,...,...,...,...
81,VACUUM ACCESSORIES,2,56.66,18.28
58,SLOW COOKERS,2,49.17,8.32
1,BAGS,1,15.83,6.11
76,TV ACCESSORIES,0,0.01,0.01


In [24]:
print("Reorder candidates:", len(reorder_candidates()))
print("Slow movers:", len(slow_movers()))
print("High-priority inventory:", len(high_priority_inventory()))
print("Active SOA:", len(active_soa()))

Reorder candidates: 110
Slow movers: 180
High-priority inventory: 0
Active SOA: 5


In [25]:
## Validate the business labels
print("FACT SALES — Record_Type")
print(fact_sales["Record_Type"].value_counts(dropna=False))

print("\nFACT SALES — Transaction_Status")
print(fact_sales["Transaction_Status"].value_counts(dropna=False))

print("\nINVENTORY — Reorder Recommendation")
print(inventory_decision["Reorder_Recommendation"].value_counts(dropna=False))

print("\nINVENTORY — Slow Mover Action")
print(inventory_decision["Slow_Mover_Action"].value_counts(dropna=False))

print("\nINVENTORY — Decision Priority")
print(inventory_decision["Decision_Priority"].value_counts(dropna=False))

print("\nINVENTORY — Inventory Decision")
print(inventory_decision["Inventory_Decision"].value_counts(dropna=False))

FACT SALES — Record_Type
Record_Type
PRODUCT           960
SERVICE_CHARGE      6
Name: count, dtype: int64

FACT SALES — Transaction_Status
Transaction_Status
POSITIVE_SALES_ACTIVITY           941
NEGATIVE_SALES_ACTIVITY            15
ZERO_UNIT_FINANCIAL_ADJUSTMENT     10
Name: count, dtype: int64

INVENTORY — Reorder Recommendation
Reorder_Recommendation
NO ACTION                  1753
REVIEW                      806
REORDER - HIGH PRIORITY      59
REORDER                      51
MONITOR CUSTOMER ORDER       12
Name: count, dtype: int64

INVENTORY — Slow Mover Action
Slow_Mover_Action
NO ACTION                      2150
MONITOR                         351
PROMOTION / MARKDOWN REVIEW     155
SLOW MOVER REVIEW                25
Name: count, dtype: int64

INVENTORY — Decision Priority
Decision_Priority
9    1222
7     806
8     351
5     155
2      59
3      51
6      25
4      12
Name: count, dtype: int64

INVENTORY — Inventory Decision
Inventory_Decision
NO ACTION                 1222


In [26]:
display(
    fact_sales[
        fact_sales["Description"]
        .astype(str)
        .str.contains(
            "INSTALL|DELIVERY|MISC",
            case=False,
            na=False
        )
    ][
        [
            "Stock Code",
            "Description",
            "Category",
            "Record_Type",
            "Transaction_Status",
            "Sold Period",
            "Sales Value",
            "Profit"
        ]
    ].head(20)
)

,Stock Code,Description,Category,Record_Type,Transaction_Status,Sold Period,Sales Value,Profit
59,DELIVERY-CHLOCAL,DELIVERY CHARGE,DELIVERY CHARGE,PRODUCT,POSITIVE_SALES_ACTIVITY,1,25.00,25.00
60,Delivery,WEB DELIVERY CHARGE,DELIVERY CHARGE,SERVICE_CHARGE,ZERO_UNIT_FINANCIAL_ADJUSTMENT,0,-0.04,-0.04
61,DELIVERY,Web Delivery Charge,DELIVERY CHARGE,SERVICE_CHARGE,POSITIVE_SALES_ACTIVITY,1,16.67,16.66
150,INSTALLATIO,INSTALLATION FEE,INSTALLATION,PRODUCT,POSITIVE_SALES_ACTIVITY,7,308.33,308.26
438,DELIVERY-CHLOCAL,DELIVERY CHARGE,DELIVERY CHARGE,PRODUCT,POSITIVE_SALES_ACTIVITY,3,34.17,34.17
439,Delivery,WEB DELIVERY CHARGE,DELIVERY CHARGE,SERVICE_CHARGE,POSITIVE_SALES_ACTIVITY,5,10.01,10.01
440,DELIVERY,Web Delivery Charge,DELIVERY CHARGE,SERVICE_CHARGE,POSITIVE_SALES_ACTIVITY,4,33.33,33.29
521,INSTALLATIO,INSTALLATION FEE,INSTALLATION,PRODUCT,POSITIVE_SALES_ACTIVITY,12,595.84,595.72
522,INSTALLATIO,INSTALLATION FREE STANDING,INSTALLATION,PRODUCT,POSITIVE_SALES_ACTIVITY,1,33.33,33.32
523,INSTALLATIO,INSTALLATION BUILT IN,INSTALLATION,PRODUCT,POSITIVE_SALES_ACTIVITY,1,66.67,66.66


#### Merchandise Sales Analytical View

Sales data contains operational records such as installation fees,
delivery charges and miscellaneous adjustments.

These records remain valid financial transactions, but they should not
be treated as merchandise products when answering questions such as:

- What is the best-selling product?
- Which products generate the most revenue?
- Which products generate the most profit?

A governed merchandise-only analytical view is therefore created for
product-ranking questions.

Financial totals may still use the complete `fact_sales` table where
appropriate.

In [28]:
NON_MERCHANDISE_CATEGORIES = [
    "INSTALLATION",
    "DELIVERY CHARGE",
    "MISC"
]

merchandise_sales = fact_sales[
    ~fact_sales["Category"].isin(NON_MERCHANDISE_CATEGORIES)
].copy()

print("Original fact_sales rows:", len(fact_sales))
print("Merchandise sales rows:", len(merchandise_sales))

Original fact_sales rows: 966
Merchandise sales rows: 950


In [30]:
def top_selling_products(n=10):

    df = merchandise_sales[
        merchandise_sales["Transaction_Status"]
        == "POSITIVE_SALES_ACTIVITY"
    ]

    return (
        df.groupby(
            ["Product_ID", "Stock Code", "Description", "Category"],
            dropna=False
        )["Sold Period"]
        .sum()
        .reset_index()
        .sort_values("Sold Period", ascending=False)
        .head(n)
    )

def top_revenue_products(n=10):

    return (
        merchandise_sales.groupby(
            ["Product_ID", "Stock Code", "Description", "Category"],
            dropna=False
        )["Sales Value"]
        .sum()
        .reset_index()
        .sort_values("Sales Value", ascending=False)
        .head(n)
    )

def top_profit_products(n=10):

    return (
        merchandise_sales.groupby(
            ["Product_ID", "Stock Code", "Description", "Category"],
            dropna=False
        )["Profit"]
        .sum()
        .reset_index()
        .sort_values("Profit", ascending=False)
        .head(n)
    )

def reorder_candidates():

    return inventory_decision[
        inventory_decision["Reorder_Recommendation"].isin(
            [
                "REORDER",
                "REORDER - HIGH PRIORITY"
            ]
        )
    ].copy()

def high_priority_reorders():

    return inventory_decision[
        inventory_decision["Reorder_Recommendation"]
        == "REORDER - HIGH PRIORITY"
    ].copy()

def slow_movers():

    return inventory_decision[
        inventory_decision["Slow_Mover_Action"]
        == "SLOW MOVER REVIEW"
    ].copy()

In [31]:
print("Reorder candidates:", len(reorder_candidates()))
print("High-priority reorders:", len(high_priority_reorders()))
print("Slow movers:", len(slow_movers()))
print("Active SOA:", len(active_soa()))

display(top_selling_products(10))

Reorder candidates: 110
High-priority reorders: 59
Slow movers: 25
Active SOA: 5


,Product_ID,Stock Code,Description,Category,Sold Period
624,1167,T2351V11,Eufy Robot Vaccum X10 Pro Omni,ROBOT CLEANING,16
692,1289,VS15A6031R4Samsung,Jet 60 Cordless Vacuum,STICK VACS,9
102,121,43LQ60006LA.LG,"43"" Smart TV",TV 33 - 43,8
506,886,P-SDU32GU18PNY,Elite microSDHC card 32G,IT ACCESSORIES,8
427,721,KN650A,Kenwood Electric Knife | KN650A,FOOD PREP,7
447,771,MC1001UK,Ninja 8-in-1 Slow Cooker,FOOD PREP,7
322,525,GN BAGS,BAGS 400/600/800 SERIES AND S5,VACUUM BAGS,7
428,722,KN650B,Kenwood Electric Kitchen Knife,FOOD PREP,5
125,170,55NANO81A6,LG 55 NANO TV,TV 51 - 59,5
482,814,MZB0JSEEU,Redmi A5 Black,TELEPHONES,5


### Step 8 — Query Router

The Query Router maps natural-language business questions to the
appropriate governed analytical function.

The router does not answer questions itself.

Instead, it classifies the request into a supported intent such as:

- Product lookup
- Top-selling products
- Revenue ranking
- Profit ranking
- Category performance
- Reorder candidates
- High-priority reorders
- Slow movers
- Active SOA

Unsupported questions are rejected rather than answered using
ungrounded assumptions.

In [33]:
def route_query(question):

    q = question.lower().strip()

    # --------------------------------------------------
    # Product lookup
    # --------------------------------------------------

    if any(term in q for term in [
        "model",
        "product lookup",
        "product details",
        "product information"
    ]):
        return "product_lookup"

    # --------------------------------------------------
    # Sales
    # --------------------------------------------------

    if any(term in q for term in [
        "top selling",
        "best selling",
        "most sold",
        "highest selling"
    ]):
        return "top_selling"

    if any(term in q for term in [
        "highest revenue",
        "top revenue",
        "most revenue"
    ]):
        return "top_revenue"

    if any(term in q for term in [
        "highest profit",
        "top profit",
        "most profitable product"
    ]):
        return "top_profit"

    if any(term in q for term in [
        "category performance",
        "best category",
        "category revenue"
    ]):
        return "category_performance"

    # --------------------------------------------------
    # Inventory
    # --------------------------------------------------

    if any(term in q for term in [
        "high priority reorder",
        "urgent reorder"
    ]):
        return "high_priority_reorder"

    if any(term in q for term in [
        "reorder",
        "replenish",
        "needs stock"
    ]):
        return "reorder"

    if any(term in q for term in [
        "slow mover",
        "slow moving",
        "slow stock"
    ]):
        return "slow_mover"

    # --------------------------------------------------
    # SOA
    # --------------------------------------------------

    if any(term in q for term in [
        "active soa",
        "current soa"
    ]):
        return "active_soa"

    return "unsupported"

def execute_query(question, n=10):

    intent = route_query(question)

    if intent == "top_selling":
        return top_selling_products(n)

    if intent == "top_revenue":
        return top_revenue_products(n)

    if intent == "top_profit":
        return top_profit_products(n)

    if intent == "category_performance":
        return category_performance()

    if intent == "reorder":
        return reorder_candidates()

    if intent == "high_priority_reorder":
        return high_priority_reorders()

    if intent == "slow_mover":
        return slow_movers()

    if intent == "active_soa":
        return active_soa()

    if intent == "product_lookup":
        return (
            "Product lookup detected. "
            "A model number is required."
        )

    return (
        "This question is not currently supported by "
        "the governed AI question set."
    )

In [34]:
test_questions = [
    "What are the top selling products?",
    "Which products generate the highest revenue?",
    "Show me products that need reordering",
    "Which products are high priority reorders?",
    "Show me slow movers",
    "Which SOA records are currently active?",
    "Who is the best employee?"
]

for question in test_questions:

    print("\nQUESTION:")
    print(question)

    print("INTENT:")
    print(route_query(question))


QUESTION:
What are the top selling products?
INTENT:
top_selling

QUESTION:
Which products generate the highest revenue?
INTENT:
top_revenue

QUESTION:
Show me products that need reordering
INTENT:
reorder

QUESTION:
Which products are high priority reorders?
INTENT:
high_priority_reorder

QUESTION:
Show me slow movers
INTENT:
slow_mover

QUESTION:
Which SOA records are currently active?
INTENT:
unsupported

QUESTION:
Who is the best employee?
INTENT:
unsupported


In [35]:
## Response Formatter
def format_top_selling(df, n=5):

    if df.empty:
        return "No selling-product records are available."

    lines = ["Top-selling products:"]

    for i, row in df.head(n).iterrows():
        lines.append(
            f"- {row['Description']} "
            f"({row['Stock Code']}): "
            f"{int(row['Sold Period'])} units"
        )

    return "\n".join(lines)

def format_top_revenue(df, n=5):

    if df.empty:
        return "No revenue records are available."

    lines = ["Highest-revenue products:"]

    for i, row in df.head(n).iterrows():
        lines.append(
            f"- {row['Description']} "
            f"({row['Stock Code']}): "
            f"£{row['Sales Value']:.2f}"
        )

    return "\n".join(lines)

def format_top_profit(df, n=5):

    if df.empty:
        return "No profit records are available."

    lines = ["Highest-profit products:"]

    for i, row in df.head(n).iterrows():
        lines.append(
            f"- {row['Description']} "
            f"({row['Stock Code']}): "
            f"£{row['Profit']:.2f}"
        )

    return "\n".join(lines)

def format_reorders(df, n=10):

    if df.empty:
        return "No reorder candidates are currently available."

    lines = [
        f"{len(df)} reorder candidate(s) found."
    ]

    for _, row in df.head(n).iterrows():

        lines.append(
            f"- {row['Description']} | "
            f"{row['Store']} | "
            f"{row['Reorder_Recommendation']}"
        )

    return "\n".join(lines)

def format_slow_movers(df, n=10):

    if df.empty:
        return "No slow-moving products are currently flagged."

    lines = [
        f"{len(df)} slow-mover record(s) found."
    ]

    for _, row in df.head(n).iterrows():

        lines.append(
            f"- {row['Description']} | "
            f"{row['Store']} | "
            f"{row['Slow_Mover_Action']}"
        )

    return "\n".join(lines)

def format_active_soa(df, n=10):

    if df.empty:
        return "No active SOA records were found."

    lines = [
        f"{len(df)} active SOA record(s) found."
    ]

    for _, row in df.head(n).iterrows():

        lines.append(
            f"- {row['Description']} "
            f"({row['Model']}): "
            f"SOA £{row['SOA']:.2f}, "
            f"valid {row['Starts'].date()} "
            f"to {row['Ends'].date()}"
        )

    return "\n".join(lines)



In [36]:
def retail_assistant(question, n=5):

    intent = route_query(question)

    if intent == "top_selling":
        return format_top_selling(
            top_selling_products(n),
            n
        )

    if intent == "top_revenue":
        return format_top_revenue(
            top_revenue_products(n),
            n
        )

    if intent == "top_profit":
        return format_top_profit(
            top_profit_products(n),
            n
        )

    if intent == "reorder":
        return format_reorders(
            reorder_candidates(),
            n
        )

    if intent == "high_priority_reorder":
        return format_reorders(
            high_priority_reorders(),
            n
        )

    if intent == "slow_mover":
        return format_slow_movers(
            slow_movers(),
            n
        )

    if intent == "active_soa":
        return format_active_soa(
            active_soa(),
            n
        )

    if intent == "product_lookup":
        return (
            "Please provide a product model number "
            "for the product lookup."
        )

    return (
        "This question is not supported by the current "
        "governed AI question set."
    )

In [37]:
questions = [
    "What are the top selling products?",
    "Which products generate the highest revenue?",
    "Which products are high priority reorders?",
    "Show me slow movers",
    "Which SOA records are currently active?",
    "Who is the best employee?"
]

for q in questions:
    print("\n" + "=" * 80)
    print("QUESTION:", q)
    print(retail_assistant(q))


QUESTION: What are the top selling products?
Top-selling products:
- Eufy Robot Vaccum X10 Pro Omni (T2351V11): 16 units
- Jet 60 Cordless Vacuum (VS15A6031R4Samsung): 9 units
- 43" Smart TV (43LQ60006LA.LG): 8 units
- Elite microSDHC card 32G (P-SDU32GU18PNY): 8 units
- Kenwood Electric Knife | KN650A (KN650A): 7 units

QUESTION: Which products generate the highest revenue?
Highest-revenue products:
- Eufy Robot Vaccum X10 Pro Omni (T2351V11): £6818.30
- Novy Panorama 120 Pro 5 Zone (42120): £4000.00
- LG 65in G5 OLED TV (OLED65G54L): £3165.00
- Miele (12392780) 10kg 1400 Spin (WEK365WCS): £2832.49
- Miele 11871830 9kg Heat Pump (TEH785WP): £2764.17

QUESTION: Which products are high priority reorders?
59 reorder candidate(s) found.
- Whirlpool Integrated Tall Freezer | Cavan | REORDER - HIGH PRIORITY
- Whirlpool Integrated Tall Freezer | Dundrum | REORDER - HIGH PRIORITY
- Whirlpool Integrated Tall Freezer | Navan | REORDER - HIGH PRIORITY
- Neff N70 St/St Single Oven | Blanch | REO

### Step 9 — Source & Refresh-Date Metadata

Every AI-generated business answer must include:

- The governed source dataset used
- The refresh/reference date of that dataset
- The query intent used to generate the answer

This provides transparency and ensures that users can understand
how current and traceable each answer is.

The assistant must never present governed analytical results without
source metadata.

In [38]:
from datetime import datetime

source_metadata = {
    "top_selling": {
        "source": "fact_sales.csv",
        "layer": "Governed Sales Layer"
    },

    "top_revenue": {
        "source": "fact_sales.csv",
        "layer": "Governed Sales Layer"
    },

    "top_profit": {
        "source": "fact_sales.csv",
        "layer": "Governed Sales Layer"
    },

    "category_performance": {
        "source": "fact_sales.csv",
        "layer": "Governed Sales Layer"
    },

    "reorder": {
        "source": "inventory_decision_layer.csv",
        "layer": "Phase 5 Inventory Intelligence"
    },

    "high_priority_reorder": {
        "source": "inventory_decision_layer.csv",
        "layer": "Phase 5 Inventory Intelligence"
    },

    "slow_mover": {
        "source": "inventory_decision_layer.csv",
        "layer": "Phase 5 Inventory Intelligence"
    },

    "active_soa": {
        "source": "fact_soa.csv",
        "layer": "Governed SOA Layer"
    },

    "product_lookup": {
        "source": "dim_product.csv",
        "layer": "Governed Product Master"
    }
}

In [39]:
def get_source_metadata(intent):

    metadata = source_metadata.get(intent)

    if metadata is None:
        return None

    return {
        "Source": metadata["source"],
        "Layer": metadata["layer"],
        "Retrieved_At": datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    }

In [40]:
def retail_assistant_with_metadata(question, n=5):

    intent = route_query(question)

    answer = retail_assistant(
        question,
        n=n
    )

    metadata = get_source_metadata(intent)

    if metadata is None:
        return answer

    metadata_text = (
        "\n\n"
        f"Source: {metadata['Source']}\n"
        f"Layer: {metadata['Layer']}\n"
        f"Retrieved: {metadata['Retrieved_At']}\n"
        f"Intent: {intent}"
    )

    return answer + metadata_text

In [41]:
print(
    retail_assistant_with_metadata(
        "What are the top selling products?"
    )
)

Top-selling products:
- Eufy Robot Vaccum X10 Pro Omni (T2351V11): 16 units
- Jet 60 Cordless Vacuum (VS15A6031R4Samsung): 9 units
- 43" Smart TV (43LQ60006LA.LG): 8 units
- Elite microSDHC card 32G (P-SDU32GU18PNY): 8 units
- Kenwood Electric Knife | KN650A (KN650A): 7 units

Source: fact_sales.csv
Layer: Governed Sales Layer
Retrieved: 2026-09-12 19:43:07
Intent: top_selling


In [42]:
print(
    retail_assistant_with_metadata(
        "Which products are high priority reorders?"
    )
)

59 reorder candidate(s) found.
- Whirlpool Integrated Tall Freezer | Cavan | REORDER - HIGH PRIORITY
- Whirlpool Integrated Tall Freezer | Dundrum | REORDER - HIGH PRIORITY
- Whirlpool Integrated Tall Freezer | Navan | REORDER - HIGH PRIORITY
- Neff N70 St/St Single Oven | Blanch | REORDER - HIGH PRIORITY
- Neff N70 St/St Single Oven | Navan | REORDER - HIGH PRIORITY

Source: inventory_decision_layer.csv
Layer: Phase 5 Inventory Intelligence
Retrieved: 2026-09-12 19:43:29
Intent: high_priority_reorder


### Step 10 — Product Knowledge Layer Schema

The Product Knowledge Layer stores verified product specifications
retrieved from trusted manufacturer sources.

Each product knowledge record is linked to the governed Product Master
using `Product_ID` and model number.

The layer will store:

- Product identity
- Manufacturer
- Model number
- Product category
- Dimensions
- Capacity / size where applicable
- Energy or power information
- Key features
- Warranty information
- Source type
- Source reference
- Extraction date
- Match confidence
- Data completeness status

Only verified or high-confidence product matches should be stored.

If an official specification source cannot be found, the product record
must be marked as incomplete rather than populated with guessed values.

In [43]:
product_knowledge_columns = [
    "Product_ID",
    "Model",
    "Product_Description",
    "Category",
    "Manufacturer",

    "Width_mm",
    "Height_mm",
    "Depth_mm",

    "Capacity_Value",
    "Capacity_Unit",

    "Power_W",
    "Energy_Rating",

    "Key_Features",
    "Warranty",

    "Source_Type",
    "Source_Reference",

    "Extraction_Date",

    "Match_Confidence",
    "Data_Completeness",
    "Validation_Status"
]

product_knowledge = pd.DataFrame(
    columns=product_knowledge_columns
)

product_knowledge

,Product_ID,Model,Product_Description,Category,Manufacturer,Width_mm,Height_mm,Depth_mm,Capacity_Value,Capacity_Unit,Power_W,Energy_Rating,Key_Features,Warranty,Source_Type,Source_Reference,Extraction_Date,Match_Confidence,Data_Completeness,Validation_Status


In [44]:
VALID_MATCH_CONFIDENCE = [
    "HIGH",
    "MEDIUM",
    "LOW"
]

VALID_COMPLETENESS = [
    "FULL_SPEC",
    "PARTIAL_SPEC",
    "NO_SPEC"
]

VALID_VALIDATION_STATUS = [
    "VALIDATED",
    "REVIEW_REQUIRED",
    "REJECTED"
]

In [45]:
def create_product_knowledge_record(
    product_id,
    model,
    description,
    category,
    manufacturer=None,
    width_mm=None,
    height_mm=None,
    depth_mm=None,
    capacity_value=None,
    capacity_unit=None,
    power_w=None,
    energy_rating=None,
    key_features=None,
    warranty=None,
    source_type=None,
    source_reference=None,
    match_confidence="HIGH",
    data_completeness="PARTIAL_SPEC",
    validation_status="REVIEW_REQUIRED"
):

    return {
        "Product_ID": product_id,
        "Model": model,
        "Product_Description": description,
        "Category": category,
        "Manufacturer": manufacturer,

        "Width_mm": width_mm,
        "Height_mm": height_mm,
        "Depth_mm": depth_mm,

        "Capacity_Value": capacity_value,
        "Capacity_Unit": capacity_unit,

        "Power_W": power_w,
        "Energy_Rating": energy_rating,

        "Key_Features": key_features,
        "Warranty": warranty,

        "Source_Type": source_type,
        "Source_Reference": source_reference,

        "Extraction_Date": pd.Timestamp.today().date(),

        "Match_Confidence": match_confidence,
        "Data_Completeness": data_completeness,
        "Validation_Status": validation_status
    }

#### Manufacturer & Model Resolution

Before retrieving external product specifications, the Product
Knowledge Layer must establish a reliable manufacturer and model identity.

The resolution process will:

- Use the governed product description and model number
- Normalize common manufacturer names
- Avoid guessing when the manufacturer is unclear
- Preserve the original governed model number
- Flag unresolved manufacturer records for manual review

Only confidently resolved products should proceed automatically to
manufacturer-source retrieval.

In [46]:
MANUFACTURER_PATTERNS = {
    "Samsung": ["SAMSUNG"],
    "LG": ["LG"],
    "Miele": ["MIELE"],
    "Whirlpool": ["WHIRLPOOL"],
    "Kenwood": ["KENWOOD"],
    "Ninja": ["NINJA"],
    "Eufy": ["EUFY"],
    "Garmin": ["GARMIN"],
    "Neff": ["NEFF"],
    "Bosch": ["BOSCH"],
    "Siemens": ["SIEMENS"],
    "Dyson": ["DYSON"],
    "Shark": ["SHARK"],
    "Sony": ["SONY"],
    "Panasonic": ["PANASONIC"],
    "Liebherr": ["LIEBHERR"],
    "Novy": ["NOVY"]
}

In [47]:
def resolve_manufacturer(description, model=None):

    text = " ".join([
        str(description or ""),
        str(model or "")
    ]).upper()

    matches = []

    for manufacturer, patterns in MANUFACTURER_PATTERNS.items():

        if any(pattern in text for pattern in patterns):
            matches.append(manufacturer)

    if len(matches) == 1:
        return {
            "Manufacturer": matches[0],
            "Manufacturer_Confidence": "HIGH"
        }

    if len(matches) > 1:
        return {
            "Manufacturer": None,
            "Manufacturer_Confidence": "REVIEW_REQUIRED"
        }

    return {
        "Manufacturer": None,
        "Manufacturer_Confidence": "UNRESOLVED"
    }

In [48]:
product_resolution = dim_product[
    [
        "Product_ID",
        "Product_Key",
        "Product_Description",
        "Product_Category"
    ]
].copy()

manufacturer_results = product_resolution.apply(
    lambda row: resolve_manufacturer(
        row["Product_Description"],
        row["Product_Key"]
    ),
    axis=1
)

product_resolution["Manufacturer"] = [
    result["Manufacturer"]
    for result in manufacturer_results
]

product_resolution["Manufacturer_Confidence"] = [
    result["Manufacturer_Confidence"]
    for result in manufacturer_results
]

In [49]:
display(
    product_resolution[
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Manufacturer",
            "Manufacturer_Confidence"
        ]
    ].head(20)
)

,Product_ID,Product_Key,Product_Description,Product_Category,Manufacturer,Manufacturer_Confidence
0,1,010-02384-10,Garmin Lily Cream Gold & White,FITNESS,Garmin,HIGH
1,2,010-02784-00,Garmin Venu 3 Smartwatch - Silver,FITNESS,Garmin,HIGH
2,3,010-02784-01,Garmin Venu 3 Smartwatch - Slate,FITNESS,Garmin,HIGH
3,4,010-02839-00,"Garmin Lily 2, Cream Gold w/",FITNESS,Garmin,HIGH
4,5,01950,NUTRIBULLET PRO 4pc Starter Kit,BLENDERS,None,UNRESOLVED
5,6,10009310,Miele GGRP Gourmet Griddle Plate,WHITES ACCESSORIES,Miele,HIGH
6,7,10107860,Miele SF-AP 50 Air Clean Plus Filter,VACUUM ACCESSORIES,Miele,HIGH
7,8,10234470,Miele Nature Flacon,WHITES ACCESSORIES,Miele,HIGH
8,9,102785,"MR Equip Metallic Red, 1.5 litre, 3kw",KETTLES,None,UNRESOLVED
9,10,103414675,ZAGG Pro Keys 2-Apple iPad Pro 13,IT ACCESSORIES,None,UNRESOLVED


In [50]:
print(
    product_resolution["Manufacturer_Confidence"]
    .value_counts(dropna=False)
)

Manufacturer_Confidence
UNRESOLVED    741
HIGH          695
Name: count, dtype: int64


In [51]:
display(
    product_resolution[
        product_resolution["Manufacturer_Confidence"]
        == "UNRESOLVED"
    ][
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category"
        ]
    ].head(20)
)

,Product_ID,Product_Key,Product_Description,Product_Category
4,5,01950,NUTRIBULLET PRO 4pc Starter Kit,BLENDERS
8,9,102785,"MR Equip Metallic Red, 1.5 litre, 3kw",KETTLES
9,10,103414675,ZAGG Pro Keys 2-Apple iPad Pro 13,IT ACCESSORIES
11,12,10795780,Sensitive Ultra Phase 2,WHITES ACCESSORIES
12,13,110200,Port Designs Belize Satchel Bag 15.6,BAGS
13,14,1107129,Berghoff Coffee /Tea Plunger .60L,HOUSEHOLD
14,15,1107130,Berghoff Coffee /Tea Plunger .80L,HOUSEHOLD
15,16,112.000,AV Link 2.0M Coaxial Plug to Coaxial,ACCESSORIES
16,17,112.204,TV Arial Lead 4.0m,ACCESSORIES
17,18,112030,Flylead F Male - F Male 2.0m,CABLES


In [52]:
## Improve manufacturer resolution safely
MANUFACTURER_PATTERNS.update({
    "Nutribullet": ["NUTRIBULLET"],
    "ZAGG": ["ZAGG"],
    "Port Designs": ["PORT DESIGNS"],
    "Berghoff": ["BERGHOFF"],
    "Russell Hobbs": ["RUSSELL HOBBS"],
    "Imetec": ["IMETEC"]
})

In [53]:
def resolve_manufacturer(description, model=None):

    text = " ".join([
        str(description or ""),
        str(model or "")
    ]).upper()

    matches = []

    for manufacturer, patterns in MANUFACTURER_PATTERNS.items():

        for pattern in patterns:

            pattern_upper = pattern.upper()

            # Short brands require word boundaries
            if len(pattern_upper) <= 2:
                found = re.search(
                    rf"\b{re.escape(pattern_upper)}\b",
                    text
                )

            # Longer manufacturer names can use normal matching
            else:
                found = pattern_upper in text

            if found:
                matches.append(manufacturer)
                break

    matches = list(set(matches))

    if len(matches) == 1:
        return {
            "Manufacturer": matches[0],
            "Manufacturer_Confidence": "HIGH"
        }

    if len(matches) > 1:
        return {
            "Manufacturer": None,
            "Manufacturer_Confidence": "REVIEW_REQUIRED"
        }

    return {
        "Manufacturer": None,
        "Manufacturer_Confidence": "UNRESOLVED"
    }

In [54]:
manufacturer_results = product_resolution.apply(
    lambda row: resolve_manufacturer(
        row["Product_Description"],
        row["Product_Key"]
    ),
    axis=1
)

product_resolution["Manufacturer"] = [
    result["Manufacturer"]
    for result in manufacturer_results
]

product_resolution["Manufacturer_Confidence"] = [
    result["Manufacturer_Confidence"]
    for result in manufacturer_results
]

In [55]:
print(
    product_resolution["Manufacturer_Confidence"]
    .value_counts(dropna=False)
)

Manufacturer_Confidence
HIGH          748
UNRESOLVED    688
Name: count, dtype: int64


In [56]:
display(
    product_resolution[
        product_resolution["Manufacturer_Confidence"]
        == "REVIEW_REQUIRED"
    ][
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category"
        ]
    ].head(20)
)

,Product_ID,Product_Key,Product_Description,Product_Category


### Step 11 — Official Manufacturer Source Retrieval Strategy

Only products with a high-confidence manufacturer match are eligible
for automatic specification-source retrieval.

The retrieval priority is:

1. Official manufacturer product page
2. Official manufacturer specification page
3. Official manufacturer PDF manual/spec sheet
4. If no official source is available, mark the product as unresolved

Third-party retailer listings should not be treated as the primary
specification source for the Product Knowledge Layer.

Every retrieved source must store:

- Product_ID
- Model
- Manufacturer
- Source URL
- Source type
- Retrieval date
- Retrieval status

No specification values should be stored until the source has been
validated as belonging to the exact product model.

In [57]:
retrieval_queue = product_resolution[
    product_resolution["Manufacturer_Confidence"] == "HIGH"
][
    [
        "Product_ID",
        "Product_Key",
        "Product_Description",
        "Product_Category",
        "Manufacturer"
    ]
].copy()

retrieval_queue.rename(
    columns={
        "Product_Key": "Model",
        "Product_Description": "Description",
        "Product_Category": "Category"
    },
    inplace=True
)

retrieval_queue["Retrieval_Status"] = "PENDING"
retrieval_queue["Source_Type"] = None
retrieval_queue["Source_URL"] = None
retrieval_queue["Retrieved_At"] = None

print("Products eligible for source retrieval:", len(retrieval_queue))

display(retrieval_queue.head(20))

Products eligible for source retrieval: 748


,Product_ID,Model,Description,Category,Manufacturer,Retrieval_Status,Source_Type,Source_URL,Retrieved_At
0,1,010-02384-10,Garmin Lily Cream Gold & White,FITNESS,Garmin,PENDING,None,None,None
1,2,010-02784-00,Garmin Venu 3 Smartwatch - Silver,FITNESS,Garmin,PENDING,None,None,None
2,3,010-02784-01,Garmin Venu 3 Smartwatch - Slate,FITNESS,Garmin,PENDING,None,None,None
3,4,010-02839-00,"Garmin Lily 2, Cream Gold w/",FITNESS,Garmin,PENDING,None,None,None
4,5,01950,NUTRIBULLET PRO 4pc Starter Kit,BLENDERS,Nutribullet,PENDING,None,None,None
5,6,10009310,Miele GGRP Gourmet Griddle Plate,WHITES ACCESSORIES,Miele,PENDING,None,None,None
6,7,10107860,Miele SF-AP 50 Air Clean Plus Filter,VACUUM ACCESSORIES,Miele,PENDING,None,None,None
7,8,10234470,Miele Nature Flacon,WHITES ACCESSORIES,Miele,PENDING,None,None,None
9,10,103414675,ZAGG Pro Keys 2-Apple iPad Pro 13,IT ACCESSORIES,ZAGG,PENDING,None,None,None
10,11,107833-01,Dyson Supersonic Hair Dryer,NaN,Dyson,PENDING,None,None,None


In [58]:
display(
    retrieval_queue["Manufacturer"]
    .value_counts()
    .rename_axis("Manufacturer")
    .reset_index(name="Products")
)

,Manufacturer,Products
0,Samsung,149
1,LG,117
2,Bosch,61
3,Neff,60
4,Ninja,58
5,Russell Hobbs,51
6,Miele,48
7,Shark,37
8,Siemens,32
9,Sony,31


In [60]:
top_manufacturers = (
    retrieval_queue["Manufacturer"]
    .value_counts()
    .head(10)
)

top_manufacturers

Manufacturer
Samsung          149
LG               117
Bosch             61
Neff              60
Ninja             58
Russell Hobbs     51
Miele             48
Shark             37
Siemens           32
Sony              31
Name: count, dtype: int64

### Step 12 — Manufacturer-Specific Source Retrieval

Manufacturer websites use different URL structures, search mechanisms,
product pages and document repositories.

Therefore, product-specification retrieval will not rely on one generic
scraper.

Instead, the Product Knowledge Layer will use a manufacturer-specific
retrieval architecture.

### Initial Priority Manufacturers

Retrieval logic will initially support the manufacturers that provide
the greatest catalogue coverage:

1. Samsung
2. LG
3. Bosch
4. Neff
5. Ninja
6. Russell Hobbs
7. Miele
8. Shark
9. Siemens
10. Sony

These manufacturers represent approximately 86% of products currently
eligible for automated Product Knowledge retrieval.

### Retrieval Principle

For each model:

1. Identify the manufacturer
2. Generate an official-source search query
3. Find a candidate official product/manual page
4. Validate that the returned page contains the exact model
5. Record the source
6. Only then allow specification extraction

A candidate page must never be treated as verified purely because it
contains a similar product description.

In [61]:
manufacturer_config = {
    "Samsung": {
        "priority": 1,
        "official_domain": "samsung.com"
    },

    "LG": {
        "priority": 2,
        "official_domain": "lg.com"
    },

    "Bosch": {
        "priority": 3,
        "official_domain": "bosch-home.co.uk"
    },

    "Neff": {
        "priority": 4,
        "official_domain": "neff-home.com"
    },

    "Ninja": {
        "priority": 5,
        "official_domain": "ninjakitchen.co.uk"
    },

    "Russell Hobbs": {
        "priority": 6,
        "official_domain": "uk.russellhobbs.com"
    },

    "Miele": {
        "priority": 7,
        "official_domain": "miele.co.uk"
    },

    "Shark": {
        "priority": 8,
        "official_domain": "sharkclean.co.uk"
    },

    "Siemens": {
        "priority": 9,
        "official_domain": "siemens-home.bsh-group.com"
    },

    "Sony": {
        "priority": 10,
        "official_domain": "sony.co.uk"
    }
}

In [62]:
retrieval_queue["Official_Domain"] = (
    retrieval_queue["Manufacturer"]
    .map(
        lambda x: manufacturer_config.get(x, {})
        .get("official_domain")
    )
)

retrieval_queue["Retrieval_Priority"] = (
    retrieval_queue["Manufacturer"]
    .map(
        lambda x: manufacturer_config.get(x, {})
        .get("priority")
    )
)

In [63]:
retrieval_queue["Official_Domain"] = (
    retrieval_queue["Manufacturer"]
    .map(
        lambda x: manufacturer_config.get(x, {})
        .get("official_domain")
    )
)

retrieval_queue["Retrieval_Priority"] = (
    retrieval_queue["Manufacturer"]
    .map(
        lambda x: manufacturer_config.get(x, {})
        .get("priority")
    )
)

In [65]:
def build_source_query(row):

    manufacturer = row["Manufacturer"]
    model = row["Model"]
    domain = row["Official_Domain"]

    if pd.isna(domain):
        return None

    return (
        f'site:{domain} "{model}" '
        f'{manufacturer} product specifications manual'
    )

retrieval_queue["Source_Query"] = (
    retrieval_queue.apply(
        build_source_query,
        axis=1
    )
)

In [66]:
display(
    retrieval_queue[
        [
            "Manufacturer",
            "Model",
            "Description",
            "Official_Domain",
            "Retrieval_Priority",
            "Source_Query"
        ]
    ]
    .sort_values("Retrieval_Priority")
    .head(20)
)

,Manufacturer,Model,Description,Official_Domain,Retrieval_Priority,Source_Query
1415,Samsung,WW90T534DASAMSUNG,9KH AUTO DOSE,samsung.com,1.0,"site:samsung.com ""WW90T534DASAMSUNG"" Samsung p..."
1414,Samsung,WW90DG6U85LEU,Samsung White 9kg 1400 Spin,samsung.com,1.0,"site:samsung.com ""WW90DG6U85LEU"" Samsung produ..."
1413,Samsung,WW90DG6U85LBU1,Samsung Series 6 Graphite Washing Machine,samsung.com,1.0,"site:samsung.com ""WW90DG6U85LBU1"" Samsung prod..."
1412,Samsung,WW90DG6U85LBU,Samsung Series 6 9kg 1400 Spin,samsung.com,1.0,"site:samsung.com ""WW90DG6U85LBU"" Samsung produ..."
1311,Samsung,WD90DG5G34BBE,Samsung Series 5 9kg/5kg,samsung.com,1.0,"site:samsung.com ""WD90DG5G34BBE"" Samsung produ..."
1323,Samsung,WF90F09C4SU1,Samsung Series 9 9kg Washing,samsung.com,1.0,"site:samsung.com ""WF90F09C4SU1"" Samsung produc..."
1280,Samsung,V11T69C23,Samsung (E71025397)Tab A9,samsung.com,1.0,"site:samsung.com ""V11T69C23"" Samsung product s..."
1288,Samsung,VS15A6031R4SAMSUNG,Jet 60 Cordless Vacuum,samsung.com,1.0,"site:samsung.com ""VS15A6031R4SAMSUNG"" Samsung ..."
1289,Samsung,VS20C8524TBSAMSUNG,Jet 85 Cordless Vac,samsung.com,1.0,"site:samsung.com ""VS20C8524TBSAMSUNG"" Samsung ..."
1290,Samsung,VS20C9547TBSAMSUNG,Jet 95 Cordless Vac,samsung.com,1.0,"site:samsung.com ""VS20C9547TBSAMSUNG"" Samsung ..."


In [67]:
## Create retrieval statuses
RETRIEVAL_STATUSES = [
    "PENDING",
    "CANDIDATE_FOUND",
    "MODEL_VERIFIED",
    "SOURCE_NOT_FOUND",
    "MODEL_MISMATCH",
    "REVIEW_REQUIRED"
]
retrieval_queue["Candidate_URL"] = None
retrieval_queue["Model_Verified"] = False
retrieval_queue["Verification_Method"] = None
retrieval_queue["Retrieval_Notes"] = None

### Step 12 — Pilot Official-Source Retrieval & Exact Model Validation

Before scaling manufacturer-source retrieval across the complete
catalogue, the retrieval workflow will be validated on a small pilot.

The pilot will use:

- 5 Samsung products
- 5 LG products

The objective is to validate:

1. Manufacturer identification
2. Model-number search query generation
3. Official-source discovery
4. Exact model-number validation
5. Source acceptance/rejection logic
6. Retrieval audit status

Only an official page containing the exact normalized model number
should receive `MODEL_VERIFIED` status.

A similar product, family page, or regional variant must not be
automatically accepted.

In [68]:
pilot_samsung = (
    retrieval_queue[
        retrieval_queue["Manufacturer"] == "Samsung"
    ]
    .head(5)
)

pilot_lg = (
    retrieval_queue[
        retrieval_queue["Manufacturer"] == "LG"
    ]
    .head(5)
)

retrieval_pilot = pd.concat(
    [
        pilot_samsung,
        pilot_lg
    ],
    ignore_index=True
).copy()

display(
    retrieval_pilot[
        [
            "Product_ID",
            "Manufacturer",
            "Model",
            "Description",
            "Source_Query"
        ]
    ]
)

,Product_ID,Manufacturer,Model,Description,Source_Query
0,314,Samsung,BRB70F26DES0EU,Samsung Integrated Series 6,"site:samsung.com ""BRB70F26DES0EU"" Samsung prod..."
1,315,Samsung,BRR29600EWW/E,Samsung Integrated Tall Larder,"site:samsung.com ""BRR29600EWW/E"" Samsung produ..."
2,316,Samsung,BRR29723EWW/EU,Samsung Built In Larder Fridge,"site:samsung.com ""BRR29723EWW/EU"" Samsung prod..."
3,392,Samsung,DV90BB9545GSAMSUNG,Blk Stl Series 8 9kg,"site:samsung.com ""DV90BB9545GSAMSUNG"" Samsung ..."
4,393,Samsung,DV90DB8845,Samsung Series 8 9kg Heat Pump,"site:samsung.com ""DV90DB8845"" Samsung product ..."
5,103,LG,32LQ63006LA.AEK,"LG 32"" Television","site:lg.com ""32LQ63006LA.AEK"" LG product speci..."
6,104,LG,32LQ63006LA.LG,"32"" Television","site:lg.com ""32LQ63006LA.LG"" LG product specif..."
7,121,LG,43LQ60006LA.LG,"43"" Smart TV","site:lg.com ""43LQ60006LA.LG"" LG product specif..."
8,122,LG,43NANO81A6,LG 43in NANO TV,"site:lg.com ""43NANO81A6"" LG product specificat..."
9,123,LG,43NANO81A6A.AE,LG 43in,"site:lg.com ""43NANO81A6A.AE"" LG product specif..."


In [70]:
print(
    retrieval_pilot["Manufacturer"]
    .value_counts()
)

print(
    "Total pilot products:",
    len(retrieval_pilot)
)

Manufacturer
Samsung    5
LG         5
Name: count, dtype: int64
Total pilot products: 10


In [72]:
## Create exact-model validator
def verify_model_in_text(model, source_text):

    if pd.isna(model) or pd.isna(source_text):
        return False

    normalized_model = normalize_model(model)
    normalized_text = normalize_model(source_text)

    if not normalized_model:
        return False

    return normalized_model in normalized_text

verify_model_in_text(
    "OLED55G54L",
    "LG OLED evo AI G5 OLED55G54L specifications"
)

True

In [73]:
verify_model_in_text(
    "OLED55G54L",
    "LG OLED55C54LA product specifications"
)

False

In [74]:
## Source acceptance function
def validate_candidate_source(
    model,
    candidate_url,
    source_text,
    official_domain
):

    if not candidate_url:
        return {
            "status": "SOURCE_NOT_FOUND",
            "model_verified": False,
            "reason": "No candidate source URL found."
        }

    # ---------------------------------------------
    # Official domain check
    # ---------------------------------------------

    if official_domain not in candidate_url.lower():
        return {
            "status": "REVIEW_REQUIRED",
            "model_verified": False,
            "reason": "Candidate source is not from the approved manufacturer domain."
        }

    # ---------------------------------------------
    # Exact model validation
    # ---------------------------------------------

    model_verified = verify_model_in_text(
        model,
        source_text
    )

    if not model_verified:
        return {
            "status": "MODEL_MISMATCH",
            "model_verified": False,
            "reason": "Exact model number was not found in the candidate source."
        }

    return {
        "status": "MODEL_VERIFIED",
        "model_verified": True,
        "reason": "Official source contains the exact normalized model number."
    }

In [75]:
test_validation = validate_candidate_source(
    model="OLED55G54L",
    candidate_url="https://www.lg.com/uk/tvs/oled55g54l",
    source_text="""
        LG OLED evo AI G5
        Model OLED55G54L
        Product specifications
    """,
    official_domain="lg.com"
)

test_validation

{'status': 'MODEL_VERIFIED',
 'model_verified': True,
 'reason': 'Official source contains the exact normalized model number.'}

In [76]:
validate_candidate_source(
    model="OLED55G54L",
    candidate_url="https://www.lg.com/uk/tvs/oled55c54la",
    source_text="""
        LG OLED C5
        Model OLED55C54LA
    """,
    official_domain="lg.com"
)

{'status': 'MODEL_MISMATCH',
 'model_verified': False,
 'reason': 'Exact model number was not found in the candidate source.'}

In [77]:
validate_candidate_source(
    model="OLED55G54L",
    candidate_url="https://example-retailer.com/lg-oled55g54l",
    source_text="LG OLED55G54L",
    official_domain="lg.com"
)

{'status': 'REVIEW_REQUIRED',
 'model_verified': False,
 'reason': 'Candidate source is not from the approved manufacturer domain.'}

In [78]:
## Prepare the pilot for real retrieval
pilot_columns = [
    "Candidate_URL",
    "Candidate_Title",
    "Source_Text",
    "Retrieval_Status",
    "Model_Verified",
    "Verification_Method",
    "Retrieved_At"
]

for col in pilot_columns:

    if col not in retrieval_pilot.columns:
        retrieval_pilot[col] = None

In [79]:
retrieval_pilot[
    [
        "Manufacturer",
        "Model",
        "Description",
        "Official_Domain",
        "Source_Query",
        "Retrieval_Status"
    ]
]

,Manufacturer,Model,Description,Official_Domain,Source_Query,Retrieval_Status
0,Samsung,BRB70F26DES0EU,Samsung Integrated Series 6,samsung.com,"site:samsung.com ""BRB70F26DES0EU"" Samsung prod...",PENDING
1,Samsung,BRR29600EWW/E,Samsung Integrated Tall Larder,samsung.com,"site:samsung.com ""BRR29600EWW/E"" Samsung produ...",PENDING
2,Samsung,BRR29723EWW/EU,Samsung Built In Larder Fridge,samsung.com,"site:samsung.com ""BRR29723EWW/EU"" Samsung prod...",PENDING
3,Samsung,DV90BB9545GSAMSUNG,Blk Stl Series 8 9kg,samsung.com,"site:samsung.com ""DV90BB9545GSAMSUNG"" Samsung ...",PENDING
4,Samsung,DV90DB8845,Samsung Series 8 9kg Heat Pump,samsung.com,"site:samsung.com ""DV90DB8845"" Samsung product ...",PENDING
5,LG,32LQ63006LA.AEK,"LG 32"" Television",lg.com,"site:lg.com ""32LQ63006LA.AEK"" LG product speci...",PENDING
6,LG,32LQ63006LA.LG,"32"" Television",lg.com,"site:lg.com ""32LQ63006LA.LG"" LG product specif...",PENDING
7,LG,43LQ60006LA.LG,"43"" Smart TV",lg.com,"site:lg.com ""43LQ60006LA.LG"" LG product specif...",PENDING
8,LG,43NANO81A6,LG 43in NANO TV,lg.com,"site:lg.com ""43NANO81A6"" LG product specificat...",PENDING
9,LG,43NANO81A6A.AE,LG 43in,lg.com,"site:lg.com ""43NANO81A6A.AE"" LG product specif...",PENDING


In [80]:
### Test with 5 samples
pilot_sources = pd.DataFrame([
    {
        "Manufacturer": "Samsung",
        "Model": "DV90DB8845",
        "Candidate_URL": "https://www.samsung.com/it/washers-and-dryers/dryers/dv8000d-dryer-a-energy-efficiency-rating-ai-dry-plus-quickdrive-9kg-black-dv90db8845gbu3/",
        "Official_Domain": "samsung.com"
    },
    {
        "Manufacturer": "Samsung",
        "Model": "MC28H5013AS/EU",
        "Candidate_URL": "https://www.samsung.com/uk/business/microwave-ovens/convection/microwave-oven-convection-mc28h5013as-eu/",
        "Official_Domain": "samsung.com"
    },
    {
        "Manufacturer": "Samsung",
        "Model": "NV68A1170BS/EU",
        "Candidate_URL": "https://www.samsung.com/uk/cooking-appliances/ovens/nv3300a--nv68a1172rs-nv68a1170bs-eu/",
        "Official_Domain": "samsung.com"
    },
    {
        "Manufacturer": "LG",
        "Model": "32LQ63006LA.AEK",
        "Candidate_URL": "https://www.lg.com/uk/tvs-soundbars/smart-tvs/32lq63006la/",
        "Official_Domain": "lg.com"
    },
    {
        "Manufacturer": "LG",
        "Model": "43LQ60006LA.LG",
        "Candidate_URL": "https://www.lg.com/uk/support/product-support/cs-43LQ60006LA.AEKQ/",
        "Official_Domain": "lg.com"
    }
])

display(pilot_sources)

,Manufacturer,Model,Candidate_URL,Official_Domain
0,Samsung,DV90DB8845,https://www.samsung.com/it/washers-and-dryers/...,samsung.com
1,Samsung,MC28H5013AS/EU,https://www.samsung.com/uk/business/microwave-...,samsung.com
2,Samsung,NV68A1170BS/EU,https://www.samsung.com/uk/cooking-appliances/...,samsung.com
3,LG,32LQ63006LA.AEK,https://www.lg.com/uk/tvs-soundbars/smart-tvs/...,lg.com
4,LG,43LQ60006LA.LG,https://www.lg.com/uk/support/product-support/...,lg.com


In [81]:
def normalize_base_model(model):
    """
    Normalize model and remove common regional/variant suffixes
    after '.', '/' or '-' where appropriate.
    """

    if pd.isna(model):
        return None

    model = str(model).upper().strip()

    # Keep the primary model component
    model = re.split(r"[./]", model)[0]

    return re.sub(r"[^A-Z0-9]", "", model)

In [82]:
models = [
    "DV90DB8845",
    "DV90DB8845GBU3",
    "MC28H5013AS/EU",
    "NV68A1170BS/EU",
    "32LQ63006LA.AEK",
    "43LQ60006LA.LG"
]

for m in models:
    print(m, "->", normalize_base_model(m))

DV90DB8845 -> DV90DB8845
DV90DB8845GBU3 -> DV90DB8845GBU3
MC28H5013AS/EU -> MC28H5013AS
NV68A1170BS/EU -> NV68A1170BS
32LQ63006LA.AEK -> 32LQ63006LA
43LQ60006LA.LG -> 43LQ60006LA


In [83]:
def verify_product_variant(
    governed_model,
    official_model
):

    governed_full = normalize_model(governed_model)
    official_full = normalize_model(official_model)

    governed_base = normalize_base_model(governed_model)
    official_base = normalize_base_model(official_model)

    # Exact complete match
    if governed_full == official_full:
        return {
            "Match_Status": "EXACT_MATCH",
            "Confidence": "HIGH"
        }

    # Base model contained in official variant
    if (
        governed_base
        and official_full
        and governed_base in official_full
    ):
        return {
            "Match_Status": "BASE_MODEL_MATCH",
            "Confidence": "HIGH"
        }

    if (
        official_base
        and governed_full
        and official_base in governed_full
    ):
        return {
            "Match_Status": "BASE_MODEL_MATCH",
            "Confidence": "HIGH"
        }

    return {
        "Match_Status": "MODEL_MISMATCH",
        "Confidence": "LOW"
    }

In [84]:
test_pairs = [
    ("DV90DB8845", "DV90DB8845GBU3"),
    ("MC28H5013AS/EU", "MC28H5013AS/EU"),
    ("NV68A1170BS/EU", "NV68A1170BS/EU"),
    ("32LQ63006LA.AEK", "32LQ63006LA"),
    ("43LQ60006LA.LG", "43LQ60006LA.AEKQ")
]

for governed, official in test_pairs:

    print(
        governed,
        "vs",
        official,
        "->",
        verify_product_variant(governed, official)
    )

DV90DB8845 vs DV90DB8845GBU3 -> {'Match_Status': 'BASE_MODEL_MATCH', 'Confidence': 'HIGH'}
MC28H5013AS/EU vs MC28H5013AS/EU -> {'Match_Status': 'EXACT_MATCH', 'Confidence': 'HIGH'}
NV68A1170BS/EU vs NV68A1170BS/EU -> {'Match_Status': 'EXACT_MATCH', 'Confidence': 'HIGH'}
32LQ63006LA.AEK vs 32LQ63006LA -> {'Match_Status': 'BASE_MODEL_MATCH', 'Confidence': 'HIGH'}
43LQ60006LA.LG vs 43LQ60006LA.AEKQ -> {'Match_Status': 'BASE_MODEL_MATCH', 'Confidence': 'HIGH'}


In [85]:
## Structured Specification Extraction
spec_schema = {
    "Product_ID": None,
    "Model": None,
    "Manufacturer": None,
    "Product_Description": None,
    "Category": None,

    "Width_mm": None,
    "Height_mm": None,
    "Depth_mm": None,

    "Capacity_Value": None,
    "Capacity_Unit": None,

    "Power_W": None,
    "Energy_Rating": None,

    "Key_Features": None,
    "Warranty": None,

    "Source_Type": None,
    "Source_Reference": None,

    "Match_Confidence": None,
    "Data_Completeness": None,
    "Validation_Status": None
}

In [86]:
def calculate_spec_completeness(record):

    important_fields = [
        "Width_mm",
        "Height_mm",
        "Depth_mm",
        "Capacity_Value",
        "Energy_Rating",
        "Key_Features"
    ]

    available = sum(
        record.get(field) not in [None, "", []]
        for field in important_fields
    )

    ratio = available / len(important_fields)

    if ratio >= 0.8:
        return "FULL_SPEC"

    if ratio > 0:
        return "PARTIAL_SPEC"

    return "NO_SPEC"

In [87]:
def validate_extracted_spec(record):

    issues = []

    # Dimensions
    for field in [
        "Width_mm",
        "Height_mm",
        "Depth_mm"
    ]:
        value = record.get(field)

        if value is not None:
            if value <= 0 or value > 5000:
                issues.append(
                    f"Invalid {field}: {value}"
                )

    # Capacity
    capacity = record.get("Capacity_Value")

    if capacity is not None and capacity <= 0:
        issues.append(
            f"Invalid capacity: {capacity}"
        )

    # Power
    power = record.get("Power_W")

    if power is not None:
        if power <= 0 or power > 15000:
            issues.append(
                f"Invalid power value: {power}"
            )

    return {
        "valid": len(issues) == 0,
        "issues": issues
    }

In [88]:
pilot_specs = pd.DataFrame(
    columns=product_knowledge_columns
)

In [89]:
dv90_record = create_product_knowledge_record(
    product_id=None,   # we will resolve this from dim_product next
    model="DV90DB8845",
    description="Samsung Bespoke AI Silent Dry 9Kg",
    category="TUMBLE DRYERS",
    manufacturer="Samsung",

    width_mm=600,
    height_mm=850,
    depth_mm=600,

    capacity_value=9,
    capacity_unit="kg",

    power_w=None,
    energy_rating="A",

    key_features=[
        "AI Dry+",
        "AI Dry",
        "Heat Pump Drying"
    ],

    warranty=None,

    source_type="Official Manufacturer Product Page",
    source_reference="https://www.samsung.com/it/washers-and-dryers/dryers/dv8000d-dryer-a-energy-efficiency-rating-ai-dry-plus-quickdrive-9kg-black-dv90db8845gbu3/",

    match_confidence="HIGH",
    data_completeness="PARTIAL_SPEC",
    validation_status="REVIEW_REQUIRED"
)

In [90]:
dv90_match = lookup_product("DV90DB8845")

dv90_match

{'status': 'matched',
 'match_confidence': 'high',
 'Product_ID': np.int64(393),
 'Product_Key': 'DV90DB8845',
 'Product_Description': 'Samsung Series 8 9kg Heat Pump',
 'Product_Category': 'TUMBLE DRYERS',
 'In_Sales': np.True_,
 'In_Stock': np.False_,
 'In_SOA': np.False_}

In [91]:
dv90_record["Product_ID"] = int(
    dv90_match["Product_ID"]
)

dv90_record["Product_Description"] = (
    dv90_match["Product_Description"]
)

dv90_record["Category"] = (
    dv90_match["Product_Category"]
)

In [92]:
dv90_record["Data_Completeness"] = (
    calculate_spec_completeness(
        dv90_record
    )
)

In [93]:
dv90_validation = validate_extracted_spec(
    dv90_record
)

dv90_validation

{'valid': True, 'issues': []}

In [94]:
if dv90_validation["valid"]:
    dv90_record["Validation_Status"] = "VALIDATED"
else:
    dv90_record["Validation_Status"] = "REVIEW_REQUIRED"

In [95]:
dv90_record

{'Product_ID': 393,
 'Model': 'DV90DB8845',
 'Product_Description': 'Samsung Series 8 9kg Heat Pump',
 'Category': 'TUMBLE DRYERS',
 'Manufacturer': 'Samsung',
 'Width_mm': 600,
 'Height_mm': 850,
 'Depth_mm': 600,
 'Capacity_Value': 9,
 'Capacity_Unit': 'kg',
 'Power_W': None,
 'Energy_Rating': 'A',
 'Key_Features': ['AI Dry+', 'AI Dry', 'Heat Pump Drying'],
 'Warranty': None,
 'Source_Type': 'Official Manufacturer Product Page',
 'Source_Reference': 'https://www.samsung.com/it/washers-and-dryers/dryers/dv8000d-dryer-a-energy-efficiency-rating-ai-dry-plus-quickdrive-9kg-black-dv90db8845gbu3/',
 'Extraction_Date': datetime.date(2026, 9, 12),
 'Match_Confidence': 'HIGH',
 'Data_Completeness': 'FULL_SPEC',
 'Validation_Status': 'VALIDATED'}

In [96]:
pilot_specs = pd.concat(
    [
        pilot_specs,
        pd.DataFrame([dv90_record])
    ],
    ignore_index=True
)

In [97]:
display(pilot_specs)

,Product_ID,Model,Product_Description,Category,Manufacturer,Width_mm,Height_mm,Depth_mm,Capacity_Value,Capacity_Unit,Power_W,Energy_Rating,Key_Features,Warranty,Source_Type,Source_Reference,Extraction_Date,Match_Confidence,Data_Completeness,Validation_Status
0,393,DV90DB8845,Samsung Series 8 9kg Heat Pump,TUMBLE DRYERS,Samsung,600,850,600,9,kg,None,A,"[AI Dry+, AI Dry, Heat Pump Drying]",None,Official Manufacturer Product Page,https://www.samsung.com/it/washers-and-dryers/...,2026-09-12,HIGH,FULL_SPEC,VALIDATED


In [98]:

# ============================================================
# BUILD + VALIDATE REMAINING 4 PILOT PRODUCTS
# ============================================================

remaining_products = [

    # --------------------------------------------------------
    # 1. Samsung Microwave
    # --------------------------------------------------------
    {
        "model": "MC28H5013AS/EU",
        "manufacturer": "Samsung",

        "width_mm": 517,
        "height_mm": 310,
        "depth_mm": 474.8,

        "capacity_value": 28,
        "capacity_unit": "L",

        "power_w": 2900,   # Maximum power consumption
        "energy_rating": None,

        "key_features": [
            "Convection cooking",
            "Dough Proof / Yogurt function",
            "Eco Mode",
            "Deodorization",
            "Ceramic enamel cavity",
            "Turntable On/Off"
        ],

        "warranty": None,

        "source_type": "Official Manufacturer Product Page",
        "source_reference":
            "https://www.samsung.com/uk/business/microwave-ovens/"
            "convection/microwave-oven-convection-mc28h5013as-eu/"
    },

    # --------------------------------------------------------
    # 2. Samsung Oven
    # --------------------------------------------------------
    {
        "model": "NV68A1170BS/EU",
        "manufacturer": "Samsung",

        "width_mm": 595,
        "height_mm": 595,
        "depth_mm": 570,

        "capacity_value": 68,
        "capacity_unit": "L",

        "power_w": 3900,   # Upper end of official output range
        "energy_rating": "A",

        "key_features": [
            "True Convection",
            "Pyrolytic Cleaning",
            "Steam Clean",
            "20 Auto Programs",
            "Child Safety Lock"
        ],

        "warranty": None,

        "source_type": "Official Manufacturer Product Page",
        "source_reference":
            "https://www.samsung.com/uk/cooking-appliances/ovens/"
            "nv3300a--nv68a1172rs-nv68a1170bs-eu/"
    },

    # --------------------------------------------------------
    # 3. LG 32-inch TV
    # --------------------------------------------------------
    {
        "model": "32LQ63006LA.AEK",
        "manufacturer": "LG",

        # Official page does not expose exact W/H/D in the
        # accessible source text, so keep these governed-null.
        "width_mm": None,
        "height_mm": None,
        "depth_mm": None,

        # TV screen size is not the same as appliance capacity.
        "capacity_value": None,
        "capacity_unit": None,

        "power_w": None,
        "energy_rating": None,

        "key_features": [
            "32-inch Full HD LED display",
            "α5 Gen 5 AI Processor",
            "AI Sound",
            "webOS Smart TV",
            "HDR10 Pro",
            "Game Optimizer",
            "ThinQ AI",
            "Apple AirPlay support"
        ],

        "warranty": None,

        "source_type": "Official Manufacturer Product Page",
        "source_reference":
            "https://www.lg.com/uk/tvs-soundbars/smart-tvs/"
            "32lq63006la/"
    },

    # --------------------------------------------------------
    # 4. LG 43-inch TV
    # --------------------------------------------------------
    {
        "model": "43LQ60006LA.LG",
        "manufacturer": "LG",

        # Support page confirms the model/manual source,
        # but do not populate unsupported specs yet.
        "width_mm": None,
        "height_mm": None,
        "depth_mm": None,

        "capacity_value": None,
        "capacity_unit": None,

        "power_w": None,
        "energy_rating": None,

        "key_features": None,
        "warranty": None,

        "source_type": "Official Manufacturer Support Page",
        "source_reference":
            "https://www.lg.com/uk/support/product-support/"
            "cs-43LQ60006LA.AEKQ/"
    }
]


# ============================================================
# CREATE, VALIDATE AND APPEND RECORDS
# ============================================================

new_records = []

for item in remaining_products:

    # ------------------------------------------
    # Match against governed Product Master
    # ------------------------------------------
    match = lookup_product(item["model"])

    if match["status"] != "matched":
        print(
            f"SKIPPED: {item['model']} "
            f"-> {match['status']}"
        )
        continue

    # ------------------------------------------
    # Create Product Knowledge record
    # ------------------------------------------
    record = create_product_knowledge_record(

        product_id=int(match["Product_ID"]),

        model=item["model"],

        description=match["Product_Description"],
        category=match["Product_Category"],

        manufacturer=item["manufacturer"],

        width_mm=item["width_mm"],
        height_mm=item["height_mm"],
        depth_mm=item["depth_mm"],

        capacity_value=item["capacity_value"],
        capacity_unit=item["capacity_unit"],

        power_w=item["power_w"],
        energy_rating=item["energy_rating"],

        key_features=item["key_features"],
        warranty=item["warranty"],

        source_type=item["source_type"],
        source_reference=item["source_reference"],

        match_confidence="HIGH",

        data_completeness="PARTIAL_SPEC",

        validation_status="REVIEW_REQUIRED"
    )

    # ------------------------------------------
    # Calculate completeness
    # ------------------------------------------
    record["Data_Completeness"] = (
        calculate_spec_completeness(record)
    )

    # ------------------------------------------
    # Numeric validation
    # ------------------------------------------
    validation = validate_extracted_spec(record)

    if validation["valid"]:
        record["Validation_Status"] = "VALIDATED"
    else:
        record["Validation_Status"] = "REVIEW_REQUIRED"

    # ------------------------------------------
    # Store record
    # ------------------------------------------
    new_records.append(record)

    print(
        item["model"],
        "->",
        record["Validation_Status"],
        "|",
        record["Data_Completeness"]
    )


# ============================================================
# APPEND TO PILOT DATASET
# ============================================================

pilot_specs = pd.concat(
    [
        pilot_specs,
        pd.DataFrame(new_records)
    ],
    ignore_index=True
)


# ============================================================
# DISPLAY FINAL 5-PRODUCT PILOT
# ============================================================

display(
    pilot_specs[
        [
            "Product_ID",
            "Model",
            "Manufacturer",
            "Product_Description",
            "Category",
            "Width_mm",
            "Height_mm",
            "Depth_mm",
            "Capacity_Value",
            "Capacity_Unit",
            "Power_W",
            "Energy_Rating",
            "Match_Confidence",
            "Data_Completeness",
            "Validation_Status"
        ]
    ]
)

print("\nTotal pilot knowledge records:", len(pilot_specs))

print("\nValidation Status:")
print(
    pilot_specs["Validation_Status"]
    .value_counts(dropna=False)
)

print("\nCompleteness:")
print(
    pilot_specs["Data_Completeness"]
    .value_counts(dropna=False)
)

MC28H5013AS/EU -> VALIDATED | FULL_SPEC
NV68A1170BS/EU -> VALIDATED | FULL_SPEC
32LQ63006LA.AEK -> VALIDATED | PARTIAL_SPEC
43LQ60006LA.LG -> VALIDATED | NO_SPEC


C:\Users\singh\AppData\Local\Temp\ipykernel_41964\1977051672.py:235: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pilot_specs = pd.concat(


,Product_ID,Model,Manufacturer,Product_Description,Category,Width_mm,Height_mm,Depth_mm,Capacity_Value,Capacity_Unit,Power_W,Energy_Rating,Match_Confidence,Data_Completeness,Validation_Status
0,393,DV90DB8845,Samsung,Samsung Series 8 9kg Heat Pump,TUMBLE DRYERS,600,850,600,9,kg,NaN,A,HIGH,FULL_SPEC,VALIDATED
1,774,MC28H5013AS/EU,Samsung,Samsung Silver 28L Microwave,MICROWAVE OVENS,517.0,310.0,474.8,28.0,L,2900.0,None,HIGH,FULL_SPEC,VALIDATED
2,834,NV68A1170BS/EU,Samsung,Samsung St/St 68L Pyroclean Oven,SINGLE OVENS,595.0,595.0,570.0,68.0,L,3900.0,A,HIGH,FULL_SPEC,VALIDATED
3,103,32LQ63006LA.AEK,LG,"LG 32"" Television",NaN,NaN,NaN,NaN,NaN,None,NaN,None,HIGH,PARTIAL_SPEC,VALIDATED
4,121,43LQ60006LA.LG,LG,"43"" Smart TV",TV 33 - 43,NaN,NaN,NaN,NaN,None,NaN,None,HIGH,NO_SPEC,VALIDATED



Total pilot knowledge records: 5

Validation Status:
Validation_Status
VALIDATED    5
Name: count, dtype: int64

Completeness:
Data_Completeness
FULL_SPEC       3
PARTIAL_SPEC    1
NO_SPEC         1
Name: count, dtype: int64


In [102]:
from pathlib import Path
import sys
import pandas as pd

# Find the pilot script from the notebook folder or project root.
pilot_dir = Path.cwd()
if not (pilot_dir / "euri_product_pilot.py").exists():
    pilot_dir = pilot_dir / "notebooks" / "Phase_8_AI Retail Assistant"

if not (pilot_dir / "euri_product_pilot.py").exists():
    raise FileNotFoundError("Cannot find euri_product_pilot.py. Check the notebook working directory.")

if str(pilot_dir) not in sys.path:
    sys.path.insert(0, str(pilot_dir))

from euri_product_pilot import run_pilot

# Loads your saved Euri key and makes five live API requests.
llm_results = run_pilot()
llm_specs_df = pd.DataFrame(llm_results)

display(
    llm_specs_df.reindex(columns=[
        "manufacturer",
        "requested_model",
        "http_status",
        "search_status",
        "match_status",
        "error",
    ])
)

Retrieving: Samsung DV90DB8845
  NO_SEARCH_EVIDENCE | response received
Retrieving: Samsung MC28H5013AS/EU
  NO_SEARCH_EVIDENCE | response received
Retrieving: Samsung NV68A1170BS/EU
  NO_SEARCH_EVIDENCE | response received
Retrieving: LG 32LQ63006LA.AEK
  NO_SEARCH_EVIDENCE | response received
Retrieving: LG 43LQ60006LA.LG
  NO_SEARCH_EVIDENCE | response received
Saved diagnostic results: d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System\notebooks\Phase_8_AI Retail Assistant\euri_pilot_results.json
Verified fields remain empty pending source review; candidate_specs is unverified.


,manufacturer,requested_model,http_status,search_status,match_status,error
0,Samsung,DV90DB8845,200,NO_SEARCH_EVIDENCE,UNVERIFIED,None
1,Samsung,MC28H5013AS/EU,200,NO_SEARCH_EVIDENCE,UNVERIFIED,None
2,Samsung,NV68A1170BS/EU,200,NO_SEARCH_EVIDENCE,UNVERIFIED,None
3,LG,32LQ63006LA.AEK,200,NO_SEARCH_EVIDENCE,UNVERIFIED,None
4,LG,43LQ60006LA.LG,200,NO_SEARCH_EVIDENCE,UNVERIFIED,None


In [ ]:
""" %run euri_product_pilot.py

import pandas as pd

llm_specs_df = pd.DataFrame(llm_results)
display(llm_specs_df[
    ["manufacturer", "requested_model", "http_status",
     "search_status", "match_status", "error"]
]) """

In [105]:
import json
from pathlib import Path
import pandas as pd

# Locate the project root from the notebook's working directory.
project_root = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (
            folder / "notebooks" / "Phase_8_AI Retail Assistant"
            / "euri_pilot_results.json"
        ).exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Cannot find euri_pilot_results.json.")

json_path = (
    project_root / "notebooks" / "Phase_8_AI Retail Assistant"
    / "euri_pilot_results.json"
)

with json_path.open(encoding="utf-8") as file:
    saved_results = json.load(file)

required_models = [
    "DV90DB8845",
    "MC28H5013AS/EU",
    "NV68A1170BS/EU",
    "32LQ63006LA.AEK",
    "43LQ60006LA.LG",
]

field_mapping = {
    "Product_Description": "product_name",
    "Category": "category",
    "Width_mm": "width_mm",
    "Height_mm": "height_mm",
    "Depth_mm": "depth_mm",
    "Capacity_Value": "capacity_value",
    "Capacity_Unit": "capacity_unit",
    "Power_W": "power_w",
    "Energy_Rating": "energy_rating",
    "Match_Confidence": "confidence",
    "Data_Completeness": "data_completeness",
}

rows = []

for result in saved_results:
    model = result.get("requested_model")

    if model not in required_models:
        continue

    specs = result.get("candidate_specs")
    specs = specs if isinstance(specs, dict) else {}

    row = {
        "Product_ID": pd.NA,
        "Model": model,
        "Manufacturer": result.get("manufacturer"),
        **{
            column: specs.get(source_field)
            for column, source_field in field_mapping.items()
        },
        "Validation_Status": "UNVERIFIED",
    }
    rows.append(row)

columns = [
    "Product_ID", "Model", "Manufacturer", "Product_Description",
    "Category", "Width_mm", "Height_mm", "Depth_mm",
    "Capacity_Value", "Capacity_Unit", "Power_W", "Energy_Rating",
    "Match_Confidence", "Data_Completeness", "Validation_Status",
]

product_specs_df = pd.DataFrame(rows, columns=columns)

# Look up Product_ID using an exact normalized manufacturer/model match.
dim_path = project_root / "data" / "processed" / "dim_product.csv"

def normalize(value):
    return "" if pd.isna(value) else str(value).strip().casefold()

if dim_path.exists():
    dim_product = pd.read_csv(dim_path, dtype=str)
    dim_product.columns = dim_product.columns.str.strip()

    needed = {"Product_ID", "Model", "Manufacturer"}

    if needed.issubset(dim_product.columns):
        id_lookup = {}

        for _, product in dim_product.iterrows():
            key = (
                normalize(product["Manufacturer"]),
                normalize(product["Model"]),
            )
            if pd.notna(product["Product_ID"]):
                id_lookup.setdefault(key, set()).add(product["Product_ID"])

        # Leave ambiguous or unmatched IDs blank.
        product_specs_df["Product_ID"] = [
            next(iter(matches)) if len(matches) == 1 else pd.NA
            for manufacturer, model in zip(
                product_specs_df["Manufacturer"],
                product_specs_df["Model"],
            )
            for matches in [
                id_lookup.get(
                    (normalize(manufacturer), normalize(model)), set()
                )
            ]
        ]
    else:
        print("Product_ID left blank: dim_product.csv has different column names.")

# Preserve the requested model order.
product_specs_df = (
    product_specs_df
    .sort_values("Model", key=lambda s: s.map({
        model: index for index, model in enumerate(required_models)
    }))
    .reset_index(drop=True)
)

missing_models = set(required_models) - set(product_specs_df["Model"])
if missing_models:
    print("Models missing from saved JSON:", sorted(missing_models))

display(product_specs_df.style.format(na_rep=""))

Product_ID left blank: dim_product.csv has different column names.


,Product_ID,Model,Manufacturer,Product_Description,Category,Width_mm,Height_mm,Depth_mm,Capacity_Value,Capacity_Unit,Power_W,Energy_Rating,Match_Confidence,Data_Completeness,Validation_Status
0,,DV90DB8845,Samsung,"Bespoke AI Heat Pump Tumble Dryer, 9 kg",Heat pump tumble dryer,600,850,600.000000,9.000000,kg,800.000000,A+++,0.910000,high,UNVERIFIED
1,,MC28H5013AS/EU,Samsung,28L Combination Microwave Oven with Grill,Combination microwave oven,517,310,474.000000,28.000000,L,2100.000000,,0.890000,0.820000,UNVERIFIED
2,,NV68A1170BS/EU,Samsung,"Built-in Oven with Dual Cook, 68L, Stainless Steel",Built-in electric oven,595,595,570.000000,68.000000,L,,A,0.920000,0.780000,UNVERIFIED
3,,32LQ63006LA.AEK,LG,LG 32-inch LQ63 Full HD Smart TV,Television,736,464,180.000000,,,,F,0.860000,0.650000,UNVERIFIED
4,,43LQ60006LA.LG,LG,43-inch LQ60 Full HD Smart TV,Television,977,575,80.800000,,,,F,0.880000,0.720000,UNVERIFIED
